
# 02_SOLVER - KRONA LTP

Solver oficial de capacidade para o projeto KRONA/LTP.

**Versão v18_LEGENDA_STATUS**: mantém a v12 e ajusta a segunda rodada de realocação para devolver a capacidade liberada para a máquina (`ALOC_REC`), validando também a capacidade do molde/ferramenta destino (`COD_FER_UNID`).

Esta versão mantém o padrão do **V1 por rateio proporcional com rastreabilidade**, porém incorpora a regra nova de **dupla restrição de capacidade**:

- limite mensal do **recurso produtivo** (`ALOC_REC`);
- limite mensal da **ferramenta** (`COD_FER_UNID`).

O versionamento do notebook é feito pelo **Git**. Por isso, o arquivo oficial permanece sem sufixo de versão.

- Notebook: `02_SOLVER.ipynb`
- Entrada: `bd_LTP_NEC_SOLVER.xlsx`
- Saída: `bd_SOLVER_v18.xlsx`
- Versão em teste: definida na variável `PIPELINE_VERSION`

---

## Premissas desta versão

- Antes do solver, a estrutura de produto é explodida de forma multinível.
- A demanda de componentes gerada pela estrutura é calculada de forma bruta e líquida.
- O estoque disponível do componente, após atendimento da demanda direta, é consumido na explosão.
- O solver roda sobre `NEC_PCS` consolidada com a necessidade explodida líquida.
- Esta versão trata o **ajuste pós-solver pai/filho** como diagnóstico de viabilidade final.
- Correção v11: filho sem roteiro não limita o pai; fica exposto como problema de cadastro/processo do cliente.
- Correção v11: produção do filho é distribuída proporcionalmente entre demanda direta e demanda de estrutura, evitando zerar o pai indevidamente.
- Correção v12: na guia `13_AJUSTE_PAI_FILHO`, quando `NEC_FILHO_POS_CORTE_PAI_PCS = 0`, o motivo passa a ser `SEM_NECESSIDADE_FILHO_POS_CORTE_PAI`; cálculo não foi alterado.
- Esta versão ainda **não** reaproveita automaticamente a capacidade liberada pelo ajuste pai/filho.
- A saída da estrutura não considera `INDICE_PERDA`, não exibe `FATOR_ACUMULADO` e não exibe `CICLO_ESTRUTURA`.
- A guia `11_NEC_EXPLODIDA_COMPONENTES` não exibe `QTD_CAMINHOS`.
- A guia `12_NEC_CONSOLIDADA_SOLVER` não exibe `QTD_ROTAS_CADASTRADAS` nem `QTD_ORIGENS_GERADORAS`.
- A coluna `NEC_EXPLODIDA_PCS` representa a necessidade explodida líquida usada pelo solver.
- Itens sem roteiro permanecem expostos como diagnóstico do cliente e não bloqueiam a execução.
- O ajuste pai/filho gera as guias `13_AJUSTE_PAI_FILHO`, `14_RESUMO_PAI_FILHO` e `15_CAPACIDADE_LIBERADA_PF`.

## Premissas originais do solver

- Processa somente o primeiro `MES_REF` da base.
- A demanda do item é `NEC_PCS`.
- `NEC_PCS` pertence ao item no mês, no nível `MES_REF + ID_PROD_UNID_FAT`.
- `NEC_PCS` não pertence ao recurso, ferramenta ou roteiro.
- `PCS_HORA` pertence ao roteiro/recurso/ferramenta e só é usado quando a demanda em peças é convertida em horas.
- Roteiros e alternativas são ordenados por `PRIOR_MATPAR` e `PRIOR_ROT`.
- Menor prioridade é melhor.
- Alternativas são permitidas; o saldo não atendido pode seguir para alternativas posteriores.
- Não existe faixa, meta ou percentual mínimo artificial.
- O corte de capacidade é feito por **rateio proporcional do estouro**.

---

## Regra de capacidade

O modelo controla capacidade em dois níveis ao mesmo tempo.

### 1. Capacidade do recurso produtivo

```text
SOMA(HR_PRODUZIR_SOLVER) por ALOC_REC <= HOR_REC
```

### 2. Capacidade mensal da ferramenta

```text
SOMA(HR_PRODUZIR_SOLVER) por COD_FER_UNID <= HOR_FER
```

Essa regra é necessária porque uma mesma ferramenta pode aparecer em mais de um recurso. Mesmo que exista capacidade em mais de uma máquina, a ferramenta não pode ultrapassar sua capacidade mensal total.

O caso que motivou esta correção foi:

```text
COD_FER_UNID = 0647B|MAT
```

---

## Regra do HOR_CAP

A capacidade oficial da alternativa é:

```text
HOR_CAP = min(HOR_REC, HOR_FER)
```

Sem fallback.

Se `HOR_REC = 0` ou `HOR_FER = 0`, então `HOR_CAP = 0`.

Alternativas com `HOR_CAP <= 0` não entram no plano de produção, mas permanecem nas guias de auditoria e diagnóstico.

Importante: `HOR_CAP` continua sendo mantido para auditoria da alternativa. Porém o motor também controla os saldos acumulados de `ALOC_REC` e `COD_FER_UNID` durante a alocação.

---

## Regra de rateio proporcional

Em cada rodada de prioridade, o motor pega os itens com saldo pendente e tenta alocá-los nas alternativas válidas daquele nível.

Para cada tentativa, a capacidade disponível é o menor saldo entre recurso e ferramenta:

```text
CAP_DISPONIVEL_LINHA = min(CAP_RESTANTE_ALOC_REC, CAP_RESTANTE_COD_FER_UNID)
```

Quando a demanda em horas ultrapassa a capacidade disponível, o corte é proporcional:

```text
FATOR_RATEIO = CAP_DISPONIVEL_LINHA / HR_DEMANDA_TOTAL_LIMITANTE
```

Assim, os itens que disputam o mesmo gargalo pagam proporcionalmente o estouro.

Exemplo:

```text
Demanda total = 600 h
Capacidade disponível = 500 h
Fator = 500 / 600 = 0,8333
```

Leitura:

```text
Atende 83,33%
Corta 16,67%
```

---

## Ordem de processamento

A ordem das alternativas respeita:

```text
PRIOR_MATPAR crescente
PRIOR_ROT crescente
```

Exemplo:

```text
1º PRIOR_MATPAR = 1 / PRIOR_ROT = 1
2º PRIOR_MATPAR = 1 / PRIOR_ROT = 2
3º PRIOR_MATPAR = 2 / PRIOR_ROT = 1
4º PRIOR_MATPAR = 2 / PRIOR_ROT = 2
```

O saldo não atendido de uma rodada segue para a próxima alternativa válida do item.

---

## Alternativas inválidas

Uma alternativa não entra no plano se tiver:

```text
HOR_REC <= 0
HOR_FER <= 0
HOR_CAP <= 0
PCS_HORA <= 0
```

Mesmo inválida, ela permanece nas guias de auditoria e diagnóstico, para explicar se o item não tinha alternativa real, se tinha alternativa sem capacidade, ou se tinha produtividade inválida.

---

## Não atendimento

O não atendimento final é controlado em peças:

```text
QTD_NAO_ATEND_SOLVER
```

Não calcular gap final em horas por item, porque `PCS_HORA` pertence à alternativa produtiva, não ao produto. Se o item não foi atendido, ele não está associado a uma produtividade única.

---

## Rastreabilidade inteligente

O modelo gera logs para responder perguntas como:

- Por que este item produziu só em uma máquina?
- Por que o saldo não foi para outra alternativa?
- Quem consumiu a capacidade antes?
- O corte aconteceu por falta de recurso ou por falta de ferramenta?
- A alternativa estava inválida ou apenas sem saldo disponível?

A guia principal para isso é:

```text
07_RASTRO_RATEIO
```

Ela registra, por tentativa:

- rodada de prioridade;
- item;
- recurso;
- ferramenta;
- saldo pendente antes;
- demanda em horas;
- saldo restante do recurso;
- saldo restante da ferramenta;
- capacidade disponível da linha;
- fator de rateio;
- motivo limitante;
- quantidade produzida;
- saldo pendente depois;
- status da decisão.

---

## Saídas

A planilha final deve conter:

1. `01_RESUMO_RECURSOS`
2. `02_RESUMO_FERRAMENTAS`
3. `03_PLANO_PRODUCAO`
4. `04_NAO_ATEND_ITEM`
5. `05_AUDITORIA_CAPACIDADE`
6. `06_ALTERNATIVAS_ITEM`
7. `07_RASTRO_RATEIO`
8. `08_DIAGNOSTICO_ITEM`
9. `09_DIAGNOSTICO_ALTERNATIVAS`
10. `10_ESTRUTURA_EXPLODIDA`
11. `11_NEC_EXPLODIDA_COMPONENTES`
12. `12_NEC_CONSOLIDADA_SOLVER`

---

## Validações internas

O notebook calcula validações de fechamento em memória/console, mas não exporta abas de validação no Excel.

---

## Regra central do modelo

O solver deve fazer **rateio proporcional do estouro**, respeitando simultaneamente a capacidade mensal do **recurso produtivo** e a capacidade mensal da **ferramenta**, sem perder a rastreabilidade do modelo V1.



### Ajustes v14

- Removidas da guia `15_CAPACIDADE_LIBERADA_PF` as colunas `FATOR_FINAL_VIAVEL_ORIGEM` e `FATOR_AJUSTE_SOBRE_SOLVER`.
- Removidas do Excel final as guias `17_RESUMO_REALOCACAO` e `17_RESUMO_REALOCACAO_POOL`.
- Alterada a realocação: a hora liberada volta para a máquina (`MES_REF + ALOC_REC`) e pode atender outro item da mesma máquina, desde que o molde/ferramenta destino também tenha capacidade disponível.


## Alterações v18

- Corrige duplicidade em `03_PLANO_PRODUCAO` causada por merge com fatores pai/filho.
- Recalcula `01_RESUMO_RECURSOS`, `02_RESUMO_FERRAMENTAS` e `09_DIAGNOSTICO_ALTERNATIVAS` a partir do plano deduplicado.
- Mantém a regra de realocação por máquina da v15: a hora liberada volta para `MES_REF + ALOC_REC`, validando a capacidade do `COD_FER_UNID` destino.
- Remove as guias 17.
- Inclui `19_DICIONARIO_GUIAS` com o detalhamento de guias e colunas.
- Inclui `20_LEGENDA_STATUS` com a legenda dos campos de status e seus significados.
- Padroniza o nome da saída como `bd_SOLVER_v18.xlsx`, sem sufixo descritivo.


In [ ]:

# ============================================================
# 00. CONFIGURAÇÃO
# ============================================================

print("02_SOLVER_RATEIO_PROPORCIONAL_DUPLA_RESTRICAO")

try:
    from functions import *
except Exception:
    import time
    class Temporizador:
        def iniciar(self):
            self._inicio = time.time()
            print('Temporizador iniciado.')
        def finalizar(self):
            fim = time.time()
            inicio = getattr(self, '_inicio', fim)
            print(f'Tempo total: {fim - inicio:,.2f} segundos')
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
try:
    from IPython.display import display
except Exception:
    display = print
    
# Iniciando Temporizador
timer = Temporizador()
timer.iniciar()

# Versão do pipeline em teste
# A saída do solver sempre leva esta versão no nome do arquivo.
PIPELINE_VERSION = "v18"
OUTPUT_SOLVER_FILENAME = "bd_SOLVER_v18.xlsx"
ESTRUTURA_FILENAME = "estruturas.xlsx"

# Caminho padrão do projeto no Windows
OUTPUT_DIR_WINDOWS = Path(r"C:\Users\carlo\OneDrive\BC\03. Projetos Bedin\01. Krona\LTP\02_OUTPUT")
INPUT_FILE_WINDOWS = OUTPUT_DIR_WINDOWS / "bd_LTP_NEC_SOLVER.xlsx"
OUTPUT_SOLVER_FILE_WINDOWS = OUTPUT_DIR_WINDOWS / OUTPUT_SOLVER_FILENAME
ESTRUTURA_FILE_WINDOWS = OUTPUT_DIR_WINDOWS.parent / "01_INPUT" / ESTRUTURA_FILENAME

# Fallback para execução em ambiente local/sandbox
INPUT_FILE_FALLBACK = Path("/mnt/data/bd_LTP_NEC_SOLVER.xlsx")
OUTPUT_SOLVER_FILE_FALLBACK = Path("/mnt/data") / OUTPUT_SOLVER_FILENAME
ESTRUTURA_FILE_FALLBACK = Path("/mnt/data/estruturas(6).xlsx")

if INPUT_FILE_WINDOWS.exists():
    INPUT_FILE = INPUT_FILE_WINDOWS
    OUTPUT_SOLVER_FILE = OUTPUT_SOLVER_FILE_WINDOWS
    ESTRUTURA_FILE = ESTRUTURA_FILE_WINDOWS if ESTRUTURA_FILE_WINDOWS.exists() else ESTRUTURA_FILE_FALLBACK
else:
    INPUT_FILE = INPUT_FILE_FALLBACK
    OUTPUT_SOLVER_FILE = OUTPUT_SOLVER_FILE_FALLBACK
    ESTRUTURA_FILE = ESTRUTURA_FILE_FALLBACK

print("PIPELINE_VERSION:", PIPELINE_VERSION)
print("INPUT_FILE:", INPUT_FILE)
print("OUTPUT_SOLVER_FILE:", OUTPUT_SOLVER_FILE)
print("ESTRUTURA_FILE:", ESTRUTURA_FILE)

LOTE_MIN_FLAG = True
MULTIPLO_EMB_FLAG = True
TOL = 1e-7
DATA_HORA_EXECUCAO = datetime.now().strftime("%Y-%m-%d %H:%M:%S")


In [ ]:

# ============================================================
# 01. FUNÇÕES AUXILIARES
# ============================================================

def to_num(s):
    return pd.to_numeric(s, errors="coerce").fillna(0)


def ensure_columns(df: pd.DataFrame, cols, default=0):
    for c in cols:
        if c not in df.columns:
            df[c] = default
    return df


def is_valid_alt(s):
    """Converte a coluna ALTERNATIVA_VALIDA para booleano de forma segura.

    Aceita booleanos reais e textos comuns como TRUE, SIM, YES, 1, VALIDA e ALTERNATIVA_VALIDA.
    Qualquer valor vazio ou diferente disso vira False.
    """
    if isinstance(s, pd.Series):
        if str(s.dtype).lower() == "bool":
            return s.fillna(False)
        txt = s.fillna(False).astype(str).str.strip().str.upper()
        return txt.isin(["TRUE", "VERDADEIRO", "SIM", "YES", "Y", "1", "VALIDA", "VÁLIDA", "ALTERNATIVA_VALIDA"])
    return bool(s)


def calcular_nec_pcs_se_necessario(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula NEC_PCS somente se a coluna não existir na base."""
    df = df.copy()
    if "NEC_PCS" in df.columns:
        df["NEC_PCS"] = to_num(df["NEC_PCS"])
        print("NEC_PCS lido da base de entrada.")
        return df

    required = [
        "LTP_CART_ARR_MES_ANT", "LTP_CART_MES_ATUAL", "LTP_SALDO_PREV_PCS", "LTP_EST_SEG_PCS",
        "LTP_EST_INI_PCS", "LTP_EST_TRANS_PCS", "ORI_TOT_PCS", "TRIANG_TOT_PCS",
        "LTP_SALDO_PREV_PROX_MES_PCS", "LTP_COMP_NEC_PCS", "LIMIT_PCS", "LOTE_MIN", "QTD_EMB"
    ]
    df = ensure_columns(df, required, 0)
    for c in required:
        df[c] = to_num(df[c])

    base_comum = (
        df["LTP_CART_ARR_MES_ANT"] + df["LTP_CART_MES_ATUAL"] + df["LTP_SALDO_PREV_PCS"] + df["LTP_EST_SEG_PCS"]
        - df["LTP_EST_INI_PCS"] - df["LTP_EST_TRANS_PCS"] - df["ORI_TOT_PCS"] - df["TRIANG_TOT_PCS"]
    )
    mesma_reg_nao = df.get("MESMA_REG", "SIM").astype(str).str.upper().eq("NAO")
    nec = np.where(mesma_reg_nao, base_comum + df["LTP_SALDO_PREV_PROX_MES_PCS"], base_comum)
    nec = pd.Series(nec, index=df.index).clip(lower=0)
    nec = nec + df["LTP_COMP_NEC_PCS"]
    nec = np.maximum(nec, df["LIMIT_PCS"])

    if LOTE_MIN_FLAG:
        mask_lote = (nec > 0) & (df["LOTE_MIN"] > 0)
        nec = np.where(mask_lote, np.maximum(nec, df["LOTE_MIN"]), nec)
    if MULTIPLO_EMB_FLAG:
        tipo = df.get("TIPO_PROD", "").astype(str).str.upper()
        mask_emb = (nec > 0) & (df["QTD_EMB"] > 0) & (tipo.isin(["PA", "MR"]))
        nec = np.where(mask_emb, np.ceil(nec / df["QTD_EMB"]) * df["QTD_EMB"], nec)

    df["NEC_PCS"] = pd.Series(nec, index=df.index).fillna(0)
    print("NEC_PCS calculado internamente porque a base não trouxe a coluna.")
    return df


def preparar_base(df: pd.DataFrame):
    df = df.copy()
    for c in ["PRIOR_MATPAR", "PRIOR_ROT", "PCS_HORA", "HOR_REC", "HOR_FER", "NEC_PCS"]:
        if c in df.columns:
            df[c] = to_num(df[c])
    df["HOR_CAP"] = df[["HOR_REC", "HOR_FER"]].min(axis=1)
    mes_ref = df["MES_REF"].min()
    bd_mes = df[df["MES_REF"].eq(mes_ref)].copy()
    print("MES_REF processado:", mes_ref)
    print("Linhas bd_mes:", len(bd_mes))
    return bd_mes


def classificar_motivo_invalida(row):
    motivos = []
    if pd.isna(row.get("ALOC_REC")) or str(row.get("ALOC_REC", "")).strip() == "":
        motivos.append("ALOC_REC_AUSENTE")
    if pd.isna(row.get("COD_FER_UNID")) or str(row.get("COD_FER_UNID", "")).strip() == "":
        motivos.append("COD_FER_UNID_AUSENTE")
    if row.get("PCS_HORA", 0) <= TOL:
        motivos.append("PCS_HORA_ZERO_OU_INVALIDO")
    if row.get("HOR_REC", 0) <= TOL:
        motivos.append("HOR_REC_ZERO_OU_NEGATIVO")
    if row.get("HOR_FER", 0) <= TOL:
        motivos.append("HOR_FER_ZERO_OU_NEGATIVO")
    if row.get("HOR_CAP", 0) <= TOL:
        motivos.append("HOR_CAP_ZERO_OU_NEGATIVO")
    return " | ".join(dict.fromkeys(motivos)) if motivos else "ALTERNATIVA_VALIDA"


def formatar_excel(caminho_arquivo):
    wb = load_workbook(caminho_arquivo)
    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(color="FFFFFF", bold=True)
    thin_gray = Side(style="thin", color="D9E2F3")
    for ws in wb.worksheets:
        ws.freeze_panes = "A2"
        ws.sheet_view.showGridLines = False
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.border = Border(bottom=thin_gray)
        for col_idx, column_cells in enumerate(ws.columns, start=1):
            max_len = 0
            for cell in column_cells:
                if cell.value is not None:
                    max_len = max(max_len, len(str(cell.value)))
            ws.column_dimensions[get_column_letter(col_idx)].width = min(max(max_len + 2, 10), 45)
        ws.auto_filter.ref = ws.dimensions
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                if isinstance(cell.value, (int, float)):
                    cell.number_format = '#,##0.00'
    wb.save(caminho_arquivo)



# ============================================================
# 01B. FUNÇÕES - EXPLOSÃO DE ESTRUTURA COM ESTOQUE
# ============================================================

from collections import defaultdict, deque


def normalizar_codigo_serie(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
         .fillna("")
         .str.strip()
         .str.replace(r"\.0$", "", regex=True)
    )


def ler_estrutura_produto(caminho_estrutura: Path) -> pd.DataFrame:
    if not caminho_estrutura.exists():
        print("Arquivo de estrutura não encontrado. A explosão será ignorada:", caminho_estrutura)
        return pd.DataFrame()

    estrutura = pd.read_excel(
        caminho_estrutura,
        sheet_name="estruturas",
        dtype={"cod_prod_acabado": str, "cod_insumo": str, "empresa": str}
    )

    required = ["empresa", "cod_prod_acabado", "cod_insumo", "qtd_utilizada_pcs"]
    missing = [c for c in required if c not in estrutura.columns]
    if missing:
        raise ValueError(f"Colunas obrigatórias ausentes na estrutura: {missing}")

    estrutura = estrutura.copy()
    estrutura["empresa"] = estrutura["empresa"].astype("string").fillna("").str.strip()
    estrutura["cod_prod_acabado"] = normalizar_codigo_serie(estrutura["cod_prod_acabado"])
    estrutura["cod_insumo"] = normalizar_codigo_serie(estrutura["cod_insumo"])
    estrutura["qtd_utilizada_pcs"] = to_num(estrutura["qtd_utilizada_pcs"])
    estrutura = estrutura[
        (estrutura["empresa"] != "")
        & (estrutura["cod_prod_acabado"] != "")
        & (estrutura["cod_insumo"] != "")
    ].copy()

    estrutura = estrutura.drop_duplicates(["empresa", "cod_prod_acabado", "cod_insumo"], keep="first")
    print("Estrutura lida:", estrutura.shape)
    return estrutura


def calcular_estoque_componentes_pos_demanda_direta(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula estoque disponível para a explosão após consumir a demanda direta/bruta do próprio item.

    Mantém a lógica do pipeline antigo: estoque total = estoque inicial + trânsito + ORI + triangulação.
    Depois abate a demanda bruta do item, sem descontar estoque nessa demanda, para evitar usar na estrutura
    o estoque que já estaria comprometido com a necessidade direta do próprio produto.
    """
    df = df.copy()
    required = [
        "LTP_EST_INI_PCS", "LTP_EST_TRANS_PCS", "ORI_TOT_PCS", "TRIANG_TOT_PCS",
        "LTP_CART_ARR_MES_ANT", "LTP_CART_MES_ATUAL", "LTP_SALDO_PREV_PCS",
        "LTP_SALDO_PREV_PROX_MES_PCS", "LTP_EST_SEG_PCS", "LIMIT_PCS", "LOTE_MIN", "QTD_EMB"
    ]
    df = ensure_columns(df, required, 0)
    for c in required:
        df[c] = to_num(df[c])

    df["COD_PROD"] = normalizar_codigo_serie(df["COD_PROD"])
    df["UNID_PROD"] = df["UNID_PROD"].astype("string").fillna("").str.strip()
    tipo_prod = df.get("TIPO_PROD", "").astype(str).str.upper()
    mesma_reg = df.get("MESMA_REG", "SIM").astype(str).str.upper()

    estoque_total = (
        df["LTP_EST_INI_PCS"]
        + df["LTP_EST_TRANS_PCS"]
        + df["ORI_TOT_PCS"]
        + df["TRIANG_TOT_PCS"]
    )

    demanda_base = (
        df["LTP_CART_ARR_MES_ANT"]
        + df["LTP_CART_MES_ATUAL"]
        + df["LTP_SALDO_PREV_PCS"]
        + df["LTP_EST_SEG_PCS"]
    )
    mask_mesma_reg_nao = mesma_reg.eq("NAO")
    demanda_base = demanda_base.where(~mask_mesma_reg_nao, demanda_base + df["LTP_SALDO_PREV_PROX_MES_PCS"])

    demanda_bruta = np.maximum(demanda_base, df["LIMIT_PCS"])

    if LOTE_MIN_FLAG:
        mask_lote = (demanda_bruta > 0) & (df["LOTE_MIN"] > 0)
        demanda_bruta = np.where(mask_lote, np.maximum(demanda_bruta, df["LOTE_MIN"]), demanda_bruta)

    if MULTIPLO_EMB_FLAG:
        mask_emb = (demanda_bruta > 0) & (df["QTD_EMB"] > 0) & (tipo_prod.isin(["PA", "MR"]))
        demanda_bruta = np.where(mask_emb, np.ceil(demanda_bruta / df["QTD_EMB"]) * df["QTD_EMB"], demanda_bruta)

    df["ESTOQUE_TOTAL_PCS"] = estoque_total
    df["DEMANDA_BRUTA_DIRETA_PCS"] = np.maximum(demanda_bruta, 0)
    df["ESTOQUE_DISP_EXPLOSAO_PCS"] = np.maximum(df["ESTOQUE_TOTAL_PCS"] - df["DEMANDA_BRUTA_DIRETA_PCS"], 0)

    estoque = (
        df.groupby(["COD_PROD", "UNID_PROD"], as_index=False)
        .agg(
            ESTOQUE_TOTAL_PCS=("ESTOQUE_TOTAL_PCS", "max"),
            DEMANDA_BRUTA_DIRETA_PCS=("DEMANDA_BRUTA_DIRETA_PCS", "max"),
            ESTOQUE_DISP_EXPLOSAO_PCS=("ESTOQUE_DISP_EXPLOSAO_PCS", "max")
        )
    )
    return estoque


def explodir_estrutura_multinivel_com_estoque(
    bd_estrutura: pd.DataFrame,
    bd_demanda_raiz: pd.DataFrame,
    bd_estoque_componentes: pd.DataFrame,
    max_nivel: int = 30
):
    cols = [
        "MES_REF", "UNID_PROD", "ID_PROD_UNID_FAT_ORIGEM", "COD_PROD_ORIGEM",
        "NEC_RAIZ_PCS", "COD_PROD_ACAB", "COD_INSUMO", "QTD_UTIL_PCS",
        "NEC_GERADA_BRUTA_PCS", "ESTOQUE_COMP_ANTES_PCS", "ESTOQUE_CONSUMIDO_PCS",
        "NEC_GERADA_LIQ_PCS", "ESTOQUE_COMP_DEPOIS_PCS", "NIVEL", "TRILHA"
    ]
    if bd_estrutura.empty or bd_demanda_raiz.empty:
        return pd.DataFrame(columns=cols), pd.DataFrame(columns=["UNID_PROD", "COD_PROD_ORIGEM", "TRILHA", "MOTIVO"])

    estrutura_idx = defaultdict(lambda: defaultdict(list))
    for emp, pai, filho, qtd in bd_estrutura[["empresa", "cod_prod_acabado", "cod_insumo", "qtd_utilizada_pcs"]].itertuples(index=False, name=None):
        estrutura_idx[str(emp)][str(pai)].append((str(filho), float(qtd or 0)))

    estoque_dict = {
        (str(unid), str(cod)): float(est or 0)
        for cod, unid, est in bd_estoque_componentes[["COD_PROD", "UNID_PROD", "ESTOQUE_DISP_EXPLOSAO_PCS"]].itertuples(index=False, name=None)
    }

    resultados = []
    problemas = []

    ordem_cols = ["MES_REF", "UNID_PROD", "COD_PROD", "ID_PROD_UNID_FAT"]
    bd_demanda_raiz_ord = bd_demanda_raiz.sort_values([c for c in ordem_cols if c in bd_demanda_raiz.columns]).copy()

    for mes_ref, unid_prod, id_origem, cod_origem, nec_raiz in bd_demanda_raiz_ord[["MES_REF", "UNID_PROD", "ID_PROD_UNID_FAT", "COD_PROD", "NEC_PCS"]].itertuples(index=False, name=None):
        emp = str(unid_prod)
        root = str(cod_origem)
        nec_raiz = float(nec_raiz or 0)
        if nec_raiz <= TOL:
            continue
        estrutura_empresa = estrutura_idx.get(emp, {})
        if root not in estrutura_empresa:
            continue

        # Na fila entra a necessidade líquida do pai que ainda precisa ser produzida/comprada.
        fila = deque([(root, 0, nec_raiz, root, {root})])
        while fila:
            pai, nivel, nec_pai_liq, trilha, caminho = fila.popleft()
            if nivel >= max_nivel:
                problemas.append((emp, root, trilha, "MAX_NIVEL_ATINGIDO"))
                continue

            for filho, qtd_util in estrutura_empresa.get(pai, []):
                nivel_filho = nivel + 1
                trilha_filho = f"{trilha} -> {filho}"
                ciclo = filho in caminho
                nec_bruta = float(nec_pai_liq or 0) * float(qtd_util or 0)

                if ciclo:
                    problemas.append((emp, root, trilha_filho, "CICLO_ESTRUTURA"))
                    continue
                if nec_bruta <= TOL:
                    continue

                chave_estoque = (emp, str(filho))
                estoque_antes = float(estoque_dict.get(chave_estoque, 0.0))
                estoque_consumido = min(estoque_antes, nec_bruta)
                nec_liq = max(nec_bruta - estoque_consumido, 0.0)
                estoque_depois = max(estoque_antes - estoque_consumido, 0.0)
                estoque_dict[chave_estoque] = estoque_depois

                resultados.append((
                    mes_ref, emp, id_origem, root, nec_raiz, pai, filho, qtd_util,
                    nec_bruta, estoque_antes, estoque_consumido, nec_liq, estoque_depois,
                    nivel_filho, trilha_filho
                ))

                # Só explode abaixo do componente se ainda existe necessidade líquida dele.
                if nec_liq > TOL:
                    fila.append((filho, nivel_filho, nec_liq, trilha_filho, caminho | {filho}))

    bd_explodida = pd.DataFrame.from_records(resultados, columns=cols)
    bd_problemas = pd.DataFrame.from_records(problemas, columns=["UNID_PROD", "COD_PROD_ORIGEM", "TRILHA", "MOTIVO"])
    return bd_explodida, bd_problemas


def consolidar_necessidade_com_estrutura(bd_mes: pd.DataFrame, bd_estrutura_explodida: pd.DataFrame):
    chave = ["MES_REF", "COD_PROD", "UNID_PROD"]

    demanda_direta = (
        bd_mes.groupby(chave, as_index=False)
        .agg(
            NEC_DIRETA_PCS=("NEC_PCS", "max"),
            ID_PROD_UNID_FAT=("ID_PROD_UNID_FAT", "first"),
            DESC_PROD=("DESC_PROD", "first") if "DESC_PROD" in bd_mes.columns else ("COD_PROD", "first"),
            TIPO_PROD=("TIPO_PROD", "first") if "TIPO_PROD" in bd_mes.columns else ("COD_PROD", "first")
        )
    )

    if bd_estrutura_explodida.empty:
        nec_gerada = pd.DataFrame(columns=chave + ["NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS"])
    else:
        nec_gerada = (
            bd_estrutura_explodida.groupby(["MES_REF", "COD_INSUMO", "UNID_PROD"], as_index=False)
            .agg(
                NEC_EXPLODIDA_BRUTA_PCS=("NEC_GERADA_BRUTA_PCS", "sum"),
                ESTOQUE_CONSUMIDO_EXPLOSAO_PCS=("ESTOQUE_CONSUMIDO_PCS", "sum"),
                NEC_EXPLODIDA_PCS=("NEC_GERADA_LIQ_PCS", "sum")
            )
            .rename(columns={"COD_INSUMO": "COD_PROD"})
        )

    nec_consolidada = demanda_direta.merge(nec_gerada, on=chave, how="outer")
    nec_consolidada["NEC_DIRETA_PCS"] = nec_consolidada["NEC_DIRETA_PCS"].fillna(0)
    for c in ["NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS"]:
        nec_consolidada[c] = nec_consolidada[c].fillna(0)
    nec_consolidada["NEC_PCS_CONSOLIDADA"] = nec_consolidada["NEC_DIRETA_PCS"] + nec_consolidada["NEC_EXPLODIDA_PCS"]
    nec_consolidada["TEM_ROTEIRO_NO_SOLVER"] = nec_consolidada["ID_PROD_UNID_FAT"].notna()

    bd_mes_out = bd_mes.merge(
        nec_consolidada[chave + ["NEC_DIRETA_PCS", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "NEC_PCS_CONSOLIDADA"]],
        on=chave,
        how="left"
    )
    bd_mes_out["NEC_PCS_ORIGINAL"] = bd_mes_out["NEC_PCS"]
    bd_mes_out["NEC_DIRETA_PCS"] = bd_mes_out["NEC_DIRETA_PCS"].fillna(bd_mes_out["NEC_PCS_ORIGINAL"])
    for c in ["NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS"]:
        bd_mes_out[c] = bd_mes_out[c].fillna(0)
    bd_mes_out["NEC_PCS_CONSOLIDADA"] = bd_mes_out["NEC_PCS_CONSOLIDADA"].fillna(bd_mes_out["NEC_PCS_ORIGINAL"])
    bd_mes_out["NEC_PCS"] = bd_mes_out["NEC_PCS_CONSOLIDADA"]

    return bd_mes_out, nec_consolidada


# ============================================================
# 01C. FUNÇÕES - AJUSTE PÓS-SOLVER PAI/FILHO - V12 MOTIVO AJUSTADO
# ============================================================

def calcular_ajuste_pos_solver_pai_filho(estrutura_explodida: pd.DataFrame, nec_consolidada_solver: pd.DataFrame, nao_atend_item: pd.DataFrame, plano_producao: pd.DataFrame):
    """Calcula diagnóstico pós-solver de viabilidade pai/filho.

    Correção v11:
    - filho/componente sem roteiro no solver NÃO limita o pai; permanece exposto como problema de cadastro/processo;
    - filho com roteiro limita o pai pela taxa proporcional de atendimento do filho no solver;
    - a produção do filho é distribuída proporcionalmente entre demanda direta e demanda de estrutura, e não consumida 100% primeiro pela demanda direta;
    - linhas sem necessidade líquida não limitam o pai.

    Esta função ainda não reaproveita capacidade liberada; apenas calcula o diagnóstico.
    """
    cols_ajuste = [
        "MES_REF", "UNID_PROD", "ID_PROD_UNID_FAT_ORIGEM", "COD_PROD_ORIGEM", "COD_PROD_ACAB", "COD_INSUMO", "NIVEL", "TRILHA", "QTD_UTIL_PCS",
        "NEC_RAIZ_PCS", "QTD_PAI_SOLVER", "FATOR_ATEND_PAI_SOLVER", "NEC_GERADA_LIQ_PCS", "NEC_FILHO_POS_CORTE_PAI_PCS",
        "NEC_DIRETA_FILHO_PCS", "NEC_TOTAL_FILHO_SOLVER_PCS", "QTD_FILHO_SOLVER_TOTAL", "TEM_ROTEIRO_FILHO_NO_SOLVER", "FATOR_ATEND_FILHO_TOTAL",
        "FATOR_FILHO_USADO_PARA_LIMITAR", "NEC_FILHO_ATENDIDA_ESTRUTURA_PCS", "QTD_PAI_FINAL_VIAVEL_EST", "FATOR_FINAL_VIAVEL_ORIGEM", "MOTIVO_AJUSTE_PAI_FILHO"
    ]
    cols_resumo = [
        "MES_REF", "UNID_PROD", "ID_PROD_UNID_FAT_ORIGEM", "COD_PROD_ORIGEM", "NEC_RAIZ_PCS", "QTD_PAI_SOLVER", "QTD_PAI_FINAL_VIAVEL_EST",
        "QTD_REDUCAO_POS_PAI_FILHO", "FATOR_ATEND_PAI_SOLVER", "FATOR_FINAL_VIAVEL_ORIGEM", "FATOR_AJUSTE_SOBRE_SOLVER", "QTD_COMPONENTES_ANALISADOS",
        "QTD_COMPONENTES_LIMITANTES", "QTD_COMPONENTES_SEM_ROTEIRO", "MOTIVO_AJUSTE_PAI_FILHO"
    ]
    if estrutura_explodida is None or estrutura_explodida.empty:
        return pd.DataFrame(columns=cols_ajuste), pd.DataFrame(columns=cols_resumo), pd.DataFrame()

    ajuste = estrutura_explodida.copy()
    for c in ["NEC_RAIZ_PCS", "NEC_GERADA_LIQ_PCS", "QTD_UTIL_PCS"]:
        ajuste[c] = to_num(ajuste[c])

    # Produção/atendimento final por item no solver.
    atend = nao_atend_item[["MES_REF", "COD_PROD", "UNID_PROD", "QTD_ATENDIDA_SOLVER", "NEC_PCS", "NEC_DIRETA_PCS", "NEC_PCS_CONSOLIDADA"]].copy()
    for c in ["QTD_ATENDIDA_SOLVER", "NEC_PCS", "NEC_DIRETA_PCS", "NEC_PCS_CONSOLIDADA"]:
        atend[c] = to_num(atend[c])

    # Informação de roteiro do item filho pela NEC consolidada.
    if nec_consolidada_solver is not None and not nec_consolidada_solver.empty and "TEM_ROTEIRO_NO_SOLVER" in nec_consolidada_solver.columns:
        roteiro_filho = nec_consolidada_solver[["MES_REF", "COD_PROD", "UNID_PROD", "TEM_ROTEIRO_NO_SOLVER"]].copy()
        roteiro_filho = roteiro_filho.rename(columns={"COD_PROD": "COD_INSUMO", "TEM_ROTEIRO_NO_SOLVER": "TEM_ROTEIRO_FILHO_NO_SOLVER"})
    else:
        roteiro_filho = pd.DataFrame(columns=["MES_REF", "COD_INSUMO", "UNID_PROD", "TEM_ROTEIRO_FILHO_NO_SOLVER"])

    # Produção do pai/origem obtida pelo solver.
    atend_pai = atend.rename(columns={
        "COD_PROD": "COD_PROD_ORIGEM",
        "QTD_ATENDIDA_SOLVER": "QTD_PAI_SOLVER",
        "NEC_PCS": "NEC_PAI_SOLVER_PCS"
    })[["MES_REF", "UNID_PROD", "COD_PROD_ORIGEM", "QTD_PAI_SOLVER", "NEC_PAI_SOLVER_PCS"]]
    ajuste = ajuste.merge(atend_pai, on=["MES_REF", "UNID_PROD", "COD_PROD_ORIGEM"], how="left")
    ajuste["QTD_PAI_SOLVER"] = ajuste["QTD_PAI_SOLVER"].fillna(0)
    ajuste["FATOR_ATEND_PAI_SOLVER"] = np.where(ajuste["NEC_RAIZ_PCS"] > TOL, (ajuste["QTD_PAI_SOLVER"] / ajuste["NEC_RAIZ_PCS"]).clip(0, 1), 0)
    ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] = ajuste["NEC_GERADA_LIQ_PCS"] * ajuste["FATOR_ATEND_PAI_SOLVER"]

    # Produção/necessidade do filho no solver.
    atend_filho = atend.rename(columns={
        "COD_PROD": "COD_INSUMO",
        "QTD_ATENDIDA_SOLVER": "QTD_FILHO_SOLVER_TOTAL",
        "NEC_PCS": "NEC_TOTAL_FILHO_SOLVER_PCS",
        "NEC_DIRETA_PCS": "NEC_DIRETA_FILHO_PCS"
    })[["MES_REF", "UNID_PROD", "COD_INSUMO", "QTD_FILHO_SOLVER_TOTAL", "NEC_TOTAL_FILHO_SOLVER_PCS", "NEC_DIRETA_FILHO_PCS"]]
    ajuste = ajuste.merge(atend_filho, on=["MES_REF", "UNID_PROD", "COD_INSUMO"], how="left")
    ajuste = ajuste.merge(roteiro_filho, on=["MES_REF", "UNID_PROD", "COD_INSUMO"], how="left")
    for c in ["QTD_FILHO_SOLVER_TOTAL", "NEC_TOTAL_FILHO_SOLVER_PCS", "NEC_DIRETA_FILHO_PCS"]:
        ajuste[c] = ajuste[c].fillna(0)
    ajuste["TEM_ROTEIRO_FILHO_NO_SOLVER"] = ajuste["TEM_ROTEIRO_FILHO_NO_SOLVER"].fillna(False).astype(bool)

    # Fator de atendimento total do filho. Como o solver trabalha com NEC consolidada, o atendimento do filho é distribuído proporcionalmente entre demanda direta e estrutura.
    ajuste["FATOR_ATEND_FILHO_TOTAL"] = np.where(
        ajuste["NEC_TOTAL_FILHO_SOLVER_PCS"] > TOL,
        (ajuste["QTD_FILHO_SOLVER_TOTAL"] / ajuste["NEC_TOTAL_FILHO_SOLVER_PCS"]).clip(0, 1),
        1.0
    )

    # Regra v11: só limita o pai quando há necessidade líquida e o filho possui roteiro no solver.
    ajuste["FATOR_FILHO_USADO_PARA_LIMITAR"] = ajuste["FATOR_ATEND_FILHO_TOTAL"]
    ajuste.loc[ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] <= TOL, "FATOR_FILHO_USADO_PARA_LIMITAR"] = 1.0
    ajuste.loc[~ajuste["TEM_ROTEIRO_FILHO_NO_SOLVER"], "FATOR_FILHO_USADO_PARA_LIMITAR"] = 1.0
    ajuste["NEC_FILHO_ATENDIDA_ESTRUTURA_PCS"] = ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] * ajuste["FATOR_FILHO_USADO_PARA_LIMITAR"]

    ch_origem = ["MES_REF", "UNID_PROD", "ID_PROD_UNID_FAT_ORIGEM", "COD_PROD_ORIGEM"]
    fator_final_origem = ajuste.groupby(ch_origem)["FATOR_FILHO_USADO_PARA_LIMITAR"].min().reset_index(name="FATOR_FILHO_MIN")
    resumo = ajuste.groupby(ch_origem, as_index=False).agg(
        NEC_RAIZ_PCS=("NEC_RAIZ_PCS", "max"),
        QTD_PAI_SOLVER=("QTD_PAI_SOLVER", "max"),
        FATOR_ATEND_PAI_SOLVER=("FATOR_ATEND_PAI_SOLVER", "max"),
        QTD_COMPONENTES_ANALISADOS=("COD_INSUMO", "nunique")
    ).merge(fator_final_origem, on=ch_origem, how="left")
    resumo["FATOR_FILHO_MIN"] = resumo["FATOR_FILHO_MIN"].fillna(1.0)
    resumo["FATOR_FINAL_VIAVEL_ORIGEM"] = (resumo["FATOR_ATEND_PAI_SOLVER"] * resumo["FATOR_FILHO_MIN"]).clip(0, 1)
    resumo["QTD_PAI_FINAL_VIAVEL_EST"] = resumo["NEC_RAIZ_PCS"] * resumo["FATOR_FINAL_VIAVEL_ORIGEM"]
    resumo["QTD_REDUCAO_POS_PAI_FILHO"] = (resumo["QTD_PAI_SOLVER"] - resumo["QTD_PAI_FINAL_VIAVEL_EST"]).clip(lower=0)
    resumo["FATOR_AJUSTE_SOBRE_SOLVER"] = np.where(resumo["QTD_PAI_SOLVER"] > TOL, (resumo["QTD_PAI_FINAL_VIAVEL_EST"] / resumo["QTD_PAI_SOLVER"]).clip(0, 1), 0)

    limitantes = ajuste[(ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] > TOL) & (ajuste["TEM_ROTEIRO_FILHO_NO_SOLVER"]) & (ajuste["FATOR_ATEND_FILHO_TOTAL"] < 1 - TOL)]
    qtd_limitantes = limitantes.groupby(ch_origem)["COD_INSUMO"].nunique().reset_index(name="QTD_COMPONENTES_LIMITANTES") if not limitantes.empty else pd.DataFrame(columns=ch_origem + ["QTD_COMPONENTES_LIMITANTES"])
    sem_roteiro = ajuste[(ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] > TOL) & (~ajuste["TEM_ROTEIRO_FILHO_NO_SOLVER"])]
    qtd_sem_roteiro = sem_roteiro.groupby(ch_origem)["COD_INSUMO"].nunique().reset_index(name="QTD_COMPONENTES_SEM_ROTEIRO") if not sem_roteiro.empty else pd.DataFrame(columns=ch_origem + ["QTD_COMPONENTES_SEM_ROTEIRO"])
    resumo = resumo.merge(qtd_limitantes, on=ch_origem, how="left").merge(qtd_sem_roteiro, on=ch_origem, how="left")
    resumo["QTD_COMPONENTES_LIMITANTES"] = resumo["QTD_COMPONENTES_LIMITANTES"].fillna(0).astype(int)
    resumo["QTD_COMPONENTES_SEM_ROTEIRO"] = resumo["QTD_COMPONENTES_SEM_ROTEIRO"].fillna(0).astype(int)
    resumo["MOTIVO_AJUSTE_PAI_FILHO"] = np.select(
        [resumo["QTD_PAI_SOLVER"] <= TOL, resumo["FATOR_ATEND_PAI_SOLVER"] < 1 - TOL, resumo["QTD_COMPONENTES_LIMITANTES"] > 0, resumo["QTD_COMPONENTES_SEM_ROTEIRO"] > 0],
        ["PAI_NAO_PRODUZIDO_PELO_SOLVER", "PAI_CORTADO_PELO_SOLVER", "FILHO_COM_ROTEIRO_LIMITOU_PAI", "FILHO_SEM_ROTEIRO_EXPOSTO_NAO_LIMITA"],
        default="SEM_AJUSTE_POS_SOLVER"
    )

    ajuste = ajuste.merge(resumo[ch_origem + ["QTD_PAI_FINAL_VIAVEL_EST", "FATOR_FINAL_VIAVEL_ORIGEM", "MOTIVO_AJUSTE_PAI_FILHO"]], on=ch_origem, how="left")

    # Correção v12: se, após o corte do pai, não houver necessidade do filho,
    # a linha da guia 13 não pode indicar que o filho limitou o pai.
    ajuste.loc[ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] <= TOL, "MOTIVO_AJUSTE_PAI_FILHO"] = "SEM_NECESSIDADE_FILHO_POS_CORTE_PAI"

    ajuste = ajuste[[c for c in cols_ajuste if c in ajuste.columns]].copy()
    resumo = resumo[[c for c in cols_resumo if c in resumo.columns]].copy()

    # Capacidade liberada estimada: aplica somente o ajuste sobre a produção original do pai. Ainda não realimenta o solver.
    fatores = resumo.rename(columns={"ID_PROD_UNID_FAT_ORIGEM": "ID_PROD_UNID_FAT"})[["MES_REF", "ID_PROD_UNID_FAT", "FATOR_AJUSTE_SOBRE_SOLVER", "FATOR_FINAL_VIAVEL_ORIGEM"]]
    if plano_producao is not None and not plano_producao.empty:
        cap_lib = plano_producao.merge(fatores, on=["MES_REF", "ID_PROD_UNID_FAT"], how="left")
        cap_lib["FATOR_AJUSTE_SOBRE_SOLVER"] = cap_lib["FATOR_AJUSTE_SOBRE_SOLVER"].fillna(1.0)
        cap_lib["FATOR_FINAL_VIAVEL_ORIGEM"] = cap_lib["FATOR_FINAL_VIAVEL_ORIGEM"].fillna(1.0)
        cap_lib["QTD_PRODUZIR_FINAL_VIAVEL_EST"] = cap_lib["QTD_PRODUZIR_SOLVER"] * cap_lib["FATOR_AJUSTE_SOBRE_SOLVER"]
        cap_lib["QTD_REDUCAO_POS_PAI_FILHO"] = (cap_lib["QTD_PRODUZIR_SOLVER"] - cap_lib["QTD_PRODUZIR_FINAL_VIAVEL_EST"]).clip(lower=0)
        cap_lib["HR_LIBERADA_EST"] = np.where(cap_lib["PCS_HORA"] > TOL, cap_lib["QTD_REDUCAO_POS_PAI_FILHO"] / cap_lib["PCS_HORA"], 0)
        cap_lib = cap_lib[cap_lib["HR_LIBERADA_EST"] > TOL].copy()
        cap_lib = cap_lib[[c for c in ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "QTD_PRODUZIR_SOLVER", "QTD_PRODUZIR_FINAL_VIAVEL_EST", "QTD_REDUCAO_POS_PAI_FILHO", "HR_PRODUZIR_SOLVER", "HR_LIBERADA_EST", "FATOR_FINAL_VIAVEL_ORIGEM", "FATOR_AJUSTE_SOBRE_SOLVER"] if c in cap_lib.columns]].copy()
    else:
        cap_lib = pd.DataFrame(columns=["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "ALOC_REC", "COD_FER_UNID", "HR_LIBERADA_EST"])

    return ajuste, resumo, cap_lib


In [ ]:

# ============================================================
# 02. LEITURA DA BASE + EXPLOSÃO INICIAL LÍQUIDA DA ESTRUTURA
# ============================================================

bd = pd.read_excel(INPUT_FILE, dtype={"COD_PROD": str, "ID_PROD_UNID_FAT": str, "UNID_PROD": str, "UNID_FAT": str})
print("Base lida:", bd.shape)
print("Colunas:", len(bd.columns))

bd["COD_PROD"] = normalizar_codigo_serie(bd["COD_PROD"])
bd["UNID_PROD"] = bd["UNID_PROD"].astype("string").fillna("").str.strip()
if "UNID_FAT" in bd.columns:
    bd["UNID_FAT"] = bd["UNID_FAT"].astype("string").fillna("").str.strip()

bd = calcular_nec_pcs_se_necessario(bd)
bd_mes = preparar_base(bd)

estrutura_produto = ler_estrutura_produto(ESTRUTURA_FILE)
estoque_componentes = calcular_estoque_componentes_pos_demanda_direta(bd_mes)

demanda_raiz_estrutura = (
    bd_mes.sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT"])
    .groupby(["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "UNID_PROD"], as_index=False)
    .agg(NEC_PCS=("NEC_PCS", "max"))
)
demanda_raiz_estrutura = demanda_raiz_estrutura[demanda_raiz_estrutura["NEC_PCS"] > TOL].copy()

estrutura_explodida, problemas_estrutura = explodir_estrutura_multinivel_com_estoque(
    estrutura_produto,
    demanda_raiz_estrutura,
    estoque_componentes,
    max_nivel=30
)

bd_mes, nec_consolidada_solver = consolidar_necessidade_com_estrutura(bd_mes, estrutura_explodida)

nec_explodida_componentes = (
    estrutura_explodida.groupby(["MES_REF", "UNID_PROD", "COD_PROD_ORIGEM", "COD_INSUMO"], as_index=False)
    .agg(
        NEC_EXPLODIDA_BRUTA_PCS=("NEC_GERADA_BRUTA_PCS", "sum"),
        ESTOQUE_CONSUMIDO_EXPLOSAO_PCS=("ESTOQUE_CONSUMIDO_PCS", "sum"),
        NEC_EXPLODIDA_PCS=("NEC_GERADA_LIQ_PCS", "sum"),
        MENOR_NIVEL=("NIVEL", "min")
    )
    if not estrutura_explodida.empty else pd.DataFrame(columns=["MES_REF", "UNID_PROD", "COD_PROD_ORIGEM", "COD_INSUMO", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "MENOR_NIVEL"])
)

validacao_explosao = pd.DataFrame({
    "METRICA": [
        "QTD_LINHAS_ESTRUTURA_ORIGINAL",
        "QTD_PRODUTOS_RAIZ_COM_DEMANDA",
        "QTD_LINHAS_ESTRUTURA_EXPLODIDA",
        "QTD_COMPONENTES_COM_NEC_EXPLODIDA_LIQUIDA",
        "MAIOR_NIVEL_EXPLODIDO",
        "QTD_PROBLEMAS_ESTRUTURA",
        "QTD_CICLOS_ESTRUTURA",
        "NEC_DIRETA_TOTAL_PCS",
        "NEC_EXPLODIDA_BRUTA_TOTAL_PCS",
        "ESTOQUE_CONSUMIDO_EXPLOSAO_TOTAL_PCS",
        "NEC_EXPLODIDA_LIQUIDA_TOTAL_PCS",
        "NEC_CONSOLIDADA_TOTAL_PCS",
        "QTD_ITENS_COM_DEMANDA_EXPLODIDA_LIQUIDA",
        "QTD_ITENS_EXPLODIDOS_SEM_ROTEIRO_NO_SOLVER"
    ],
    "VALOR": [
        len(estrutura_produto),
        len(demanda_raiz_estrutura),
        len(estrutura_explodida),
        nec_explodida_componentes.loc[nec_explodida_componentes["NEC_EXPLODIDA_PCS"] > TOL, "COD_INSUMO"].nunique() if not nec_explodida_componentes.empty else 0,
        estrutura_explodida["NIVEL"].max() if not estrutura_explodida.empty else 0,
        len(problemas_estrutura),
        (problemas_estrutura["MOTIVO"].eq("CICLO_ESTRUTURA").sum() if not problemas_estrutura.empty else 0),
        nec_consolidada_solver["NEC_DIRETA_PCS"].sum() if not nec_consolidada_solver.empty else 0,
        nec_consolidada_solver["NEC_EXPLODIDA_BRUTA_PCS"].sum() if not nec_consolidada_solver.empty else 0,
        nec_consolidada_solver["ESTOQUE_CONSUMIDO_EXPLOSAO_PCS"].sum() if not nec_consolidada_solver.empty else 0,
        nec_consolidada_solver["NEC_EXPLODIDA_PCS"].sum() if not nec_consolidada_solver.empty else 0,
        nec_consolidada_solver["NEC_PCS_CONSOLIDADA"].sum() if not nec_consolidada_solver.empty else 0,
        (nec_consolidada_solver["NEC_EXPLODIDA_PCS"] > TOL).sum() if not nec_consolidada_solver.empty else 0,
        ((nec_consolidada_solver["NEC_EXPLODIDA_PCS"] > TOL) & (~nec_consolidada_solver["TEM_ROTEIRO_NO_SOLVER"])).sum() if not nec_consolidada_solver.empty else 0
    ]
})

print("Estoque componentes para explosão:", estoque_componentes.shape)
print("Estrutura explodida líquida:", estrutura_explodida.shape)
print("Necessidade explodida componentes:", nec_explodida_componentes.shape)
print("NEC consolidada solver:", nec_consolidada_solver.shape)
print("Linhas bd_mes após NEC consolidada:", bd_mes.shape)
display(validacao_explosao)


In [ ]:

# ============================================================
# 03. MONTAGEM DA DEMANDA E DAS ALTERNATIVAS
# ============================================================

meta_cols = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG", "NEC_PCS_ORIGINAL", "NEC_DIRETA_PCS", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "NEC_PCS_CONSOLIDADA"]

bd_demanda_solver = (
    bd_mes.sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT"])
    .groupby(["MES_REF", "ID_PROD_UNID_FAT"], as_index=False)
    .agg({**{c: "first" for c in meta_cols if c not in ["MES_REF", "ID_PROD_UNID_FAT"] and c in bd_mes.columns}, "NEC_PCS": "max"})
)
bd_demanda_solver = bd_demanda_solver[bd_demanda_solver["NEC_PCS"] > TOL].copy()

alt_cols = ["MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG", "ID_RECURSO", "ID_FERRAMENTA"]
bd_alternativas_item = bd_mes[[c for c in alt_cols if c in bd_mes.columns]].copy()
bd_alternativas_item = bd_alternativas_item[bd_alternativas_item["ID_PROD_UNID_FAT"].isin(bd_demanda_solver["ID_PROD_UNID_FAT"])].copy()
bd_alternativas_item = bd_alternativas_item.drop_duplicates([c for c in ["MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC", "COD_FER_UNID"] if c in bd_alternativas_item.columns]).reset_index(drop=True)

bd_alternativas_item["ALTERNATIVA_VALIDA"] = (
    bd_alternativas_item["ALOC_REC"].notna() & bd_alternativas_item["COD_FER_UNID"].notna()
    & (bd_alternativas_item["PCS_HORA"] > TOL) & (bd_alternativas_item["HOR_REC"] > TOL)
    & (bd_alternativas_item["HOR_FER"] > TOL) & (bd_alternativas_item["HOR_CAP"] > TOL)
)
bd_alternativas_item["MOTIVO_INVALIDA"] = bd_alternativas_item.apply(classificar_motivo_invalida, axis=1)
bd_alternativas_validas = bd_alternativas_item[bd_alternativas_item["ALTERNATIVA_VALIDA"]].copy()

bd_cap_recursos = bd_alternativas_item.dropna(subset=["ALOC_REC"]).groupby("ALOC_REC", as_index=False).agg(HOR_REC=("HOR_REC", "max"))
bd_cap_ferramentas = bd_alternativas_item.dropna(subset=["COD_FER_UNID"]).groupby("COD_FER_UNID", as_index=False).agg(HOR_FER=("HOR_FER", "max"))

print("Itens com demanda:", len(bd_demanda_solver))
print("Alternativas auditadas:", len(bd_alternativas_item))
print("Alternativas válidas para produção:", len(bd_alternativas_validas))
print("Alternativas com HOR_CAP zero:", (bd_alternativas_item["HOR_CAP"] <= TOL).sum())
print("Recursos:", len(bd_cap_recursos))
print("Ferramentas:", len(bd_cap_ferramentas))


In [ ]:

# ============================================================
# 04. SOLVER - RATEIO PROPORCIONAL COM DUPLA RESTRIÇÃO
# ============================================================

def executar_rateio_proporcional_dupla_restricao(bd_demanda, bd_alternativas_validas, bd_cap_recursos, bd_cap_ferramentas):
    alternativas = bd_alternativas_validas.copy()
    cap_restante_rec = bd_cap_recursos.set_index("ALOC_REC")["HOR_REC"].to_dict()
    cap_restante_fer = bd_cap_ferramentas.set_index("COD_FER_UNID")["HOR_FER"].to_dict()
    pendente = bd_demanda.set_index("ID_PROD_UNID_FAT")["NEC_PCS"].to_dict()
    producao, rastro = [], []

    niveis_prioridade = list(alternativas[["PRIOR_MATPAR", "PRIOR_ROT"]].drop_duplicates().sort_values(["PRIOR_MATPAR", "PRIOR_ROT"]).itertuples(index=False, name=None))
    for rodada, (prior_matpar, prior_rot) in enumerate(niveis_prioridade, start=1):
        nivel = alternativas[alternativas["PRIOR_MATPAR"].eq(prior_matpar) & alternativas["PRIOR_ROT"].eq(prior_rot)].copy()
        nivel["RODADA"] = rodada
        nivel["QTD_PENDENTE_ANTES"] = nivel["ID_PROD_UNID_FAT"].map(pendente).fillna(0)
        nivel = nivel[nivel["QTD_PENDENTE_ANTES"] > TOL].copy()
        if nivel.empty:
            continue

        qtd_alt_mesmo_nivel = nivel.groupby("ID_PROD_UNID_FAT")["ALOC_REC"].transform("count")
        nivel["QTD_ALTERNATIVAS_MESMO_NIVEL"] = qtd_alt_mesmo_nivel
        nivel["QTD_SOLICITADA"] = nivel["QTD_PENDENTE_ANTES"] / qtd_alt_mesmo_nivel
        nivel["HR_SOLICITADA"] = nivel["QTD_SOLICITADA"] / nivel["PCS_HORA"]

        nivel["CAP_RESTANTE_ALOC_REC_ANTES"] = nivel["ALOC_REC"].map(cap_restante_rec).fillna(0)
        nivel["CAP_RESTANTE_COD_FER_ANTES"] = nivel["COD_FER_UNID"].map(cap_restante_fer).fillna(0)

        demanda_rec = nivel.groupby("ALOC_REC")["HR_SOLICITADA"].transform("sum")
        demanda_fer = nivel.groupby("COD_FER_UNID")["HR_SOLICITADA"].transform("sum")
        nivel["HR_DEMANDA_TOTAL_ALOC_REC"] = demanda_rec
        nivel["HR_DEMANDA_TOTAL_COD_FER"] = demanda_fer
        nivel["FATOR_RATEIO_ALOC_REC"] = np.where(demanda_rec > TOL, np.minimum(1.0, nivel["CAP_RESTANTE_ALOC_REC_ANTES"] / demanda_rec), 0.0)
        nivel["FATOR_RATEIO_COD_FER"] = np.where(demanda_fer > TOL, np.minimum(1.0, nivel["CAP_RESTANTE_COD_FER_ANTES"] / demanda_fer), 0.0)
        nivel["FATOR_RATEIO"] = nivel[["FATOR_RATEIO_ALOC_REC", "FATOR_RATEIO_COD_FER"]].min(axis=1).clip(lower=0, upper=1)
        nivel["CAP_DISPONIVEL_LINHA"] = nivel[["CAP_RESTANTE_ALOC_REC_ANTES", "CAP_RESTANTE_COD_FER_ANTES"]].min(axis=1)

        cond_rec = nivel["FATOR_RATEIO_ALOC_REC"] < nivel["FATOR_RATEIO_COD_FER"] - TOL
        cond_fer = nivel["FATOR_RATEIO_COD_FER"] < nivel["FATOR_RATEIO_ALOC_REC"] - TOL
        cond_ambos = (~cond_rec) & (~cond_fer) & (nivel["FATOR_RATEIO"] < 1 - TOL)
        nivel["MOTIVO_LIMITANTE"] = np.select([cond_rec, cond_fer, cond_ambos], ["ALOC_REC", "COD_FER_UNID", "ALOC_REC_E_COD_FER"], default="SEM_LIMITACAO")

        nivel["HR_PRODUZIR_SOLVER"] = nivel["HR_SOLICITADA"] * nivel["FATOR_RATEIO"]
        nivel["QTD_PRODUZIR_SOLVER"] = nivel["HR_PRODUZIR_SOLVER"] * nivel["PCS_HORA"]
        nivel["QTD_PENDENTE_DEPOIS"] = (nivel["QTD_PENDENTE_ANTES"] - nivel.groupby("ID_PROD_UNID_FAT")["QTD_PRODUZIR_SOLVER"].transform("sum")).clip(lower=0)

        nivel["STATUS_DECISAO"] = np.select(
            [nivel["CAP_RESTANTE_ALOC_REC_ANTES"] <= TOL, nivel["CAP_RESTANTE_COD_FER_ANTES"] <= TOL, nivel["HR_PRODUZIR_SOLVER"] <= TOL, nivel["FATOR_RATEIO"] >= 1 - TOL, nivel["FATOR_RATEIO"] < 1 - TOL],
            ["NAO_ALOCADO_CAP_RECURSO_ZERO", "NAO_ALOCADO_CAP_FERRAMENTA_ZERO", "NAO_ALOCADO_RATEIO_ZERO", "ALOCADO_TOTAL_RODADA", "ALOCADO_PARCIAL_RATEIO"],
            default="VERIFICAR"
        )
        nivel["MOTIVO_CORTE"] = np.where(nivel["FATOR_RATEIO"] >= 1 - TOL, "SEM_CORTE", "CORTE_POR_" + nivel["MOTIVO_LIMITANTE"].astype(str))

        for aloc_rec, consumo in nivel.groupby("ALOC_REC")["HR_PRODUZIR_SOLVER"].sum().items():
            cap_restante_rec[aloc_rec] = max(0.0, cap_restante_rec.get(aloc_rec, 0.0) - float(consumo))
        for cod_fer, consumo in nivel.groupby("COD_FER_UNID")["HR_PRODUZIR_SOLVER"].sum().items():
            cap_restante_fer[cod_fer] = max(0.0, cap_restante_fer.get(cod_fer, 0.0) - float(consumo))
        for item, qtd_prod in nivel.groupby("ID_PROD_UNID_FAT")["QTD_PRODUZIR_SOLVER"].sum().items():
            pendente[item] = max(0.0, pendente.get(item, 0.0) - float(qtd_prod))

        producao.append(nivel[nivel["QTD_PRODUZIR_SOLVER"] > TOL].copy())
        rastro.append(nivel.copy())

    bd_producao = pd.concat(producao, ignore_index=True) if producao else pd.DataFrame()
    bd_rastro = pd.concat(rastro, ignore_index=True) if rastro else pd.DataFrame()
    return bd_producao, bd_rastro, pendente, cap_restante_rec, cap_restante_fer

bd_plano_producao, rastro_rateio, pendente_final, cap_restante_rec_final, cap_restante_fer_final = executar_rateio_proporcional_dupla_restricao(bd_demanda_solver, bd_alternativas_validas, bd_cap_recursos, bd_cap_ferramentas)
print("Linhas plano produção:", len(bd_plano_producao))
print("Linhas rastro rateio:", len(rastro_rateio))


In [ ]:

# ============================================================
# 05. SAÍDAS DE NEGÓCIO E RASTREABILIDADE
# ============================================================


cols_plano = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "NEC_PCS_ORIGINAL", "NEC_DIRETA_PCS", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "NEC_PCS_CONSOLIDADA", "LINHA_PROD", "FAMILIA_PROD", "MO", "RODADA", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "QTD_PENDENTE_ANTES", "QTD_SOLICITADA", "HR_SOLICITADA", "CAP_RESTANTE_ALOC_REC_ANTES", "CAP_RESTANTE_COD_FER_ANTES", "CAP_DISPONIVEL_LINHA", "HR_DEMANDA_TOTAL_ALOC_REC", "HR_DEMANDA_TOTAL_COD_FER", "FATOR_RATEIO_ALOC_REC", "FATOR_RATEIO_COD_FER", "FATOR_RATEIO", "MOTIVO_LIMITANTE", "QTD_PRODUZIR_SOLVER", "PESO_PROD_KG", "VOL_PRODUZIR_SOLVER", "HR_PRODUZIR_SOLVER", "QTD_PENDENTE_DEPOIS", "MOTIVO_CORTE", "STATUS_DECISAO", "FATOR_FINAL_VIAVEL_ORIGEM", "FATOR_AJUSTE_SOBRE_SOLVER", "QTD_PRODUZIR_FINAL_VIAVEL_EST", "QTD_REDUCAO_POS_PAI_FILHO", "HR_LIBERADA_EST"]
campos_produto_saida = ["COD_PROD", "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG"]
campos_produto_saida = [c for c in campos_produto_saida if c in bd_mes.columns]
if campos_produto_saida and "COD_PROD" in campos_produto_saida:
    cadastro_produto_saida = bd_mes[campos_produto_saida].drop_duplicates("COD_PROD")
    for c in ["LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG"]:
        if c in bd_plano_producao.columns:
            bd_plano_producao = bd_plano_producao.drop(columns=[c])
    bd_plano_producao = bd_plano_producao.merge(cadastro_produto_saida, on="COD_PROD", how="left")

bd_plano_producao["PESO_PROD_KG"] = to_num(bd_plano_producao["PESO_PROD_KG"]) if "PESO_PROD_KG" in bd_plano_producao.columns else 0
bd_plano_producao["VOL_PRODUZIR_SOLVER"] = bd_plano_producao["QTD_PRODUZIR_SOLVER"] * bd_plano_producao["PESO_PROD_KG"] if "QTD_PRODUZIR_SOLVER" in bd_plano_producao.columns else 0
plano_producao = bd_plano_producao[[c for c in cols_plano if c in bd_plano_producao.columns]].copy()

atendido_item = plano_producao.groupby(["MES_REF", "ID_PROD_UNID_FAT"], as_index=False).agg(QTD_ATENDIDA_SOLVER=("QTD_PRODUZIR_SOLVER", "sum")) if not plano_producao.empty else pd.DataFrame(columns=["MES_REF", "ID_PROD_UNID_FAT", "QTD_ATENDIDA_SOLVER"])
nao_atend_item = bd_demanda_solver.merge(atendido_item, on=["MES_REF", "ID_PROD_UNID_FAT"], how="left")
nao_atend_item["QTD_ATENDIDA_SOLVER"] = nao_atend_item["QTD_ATENDIDA_SOLVER"].fillna(0)
nao_atend_item["QTD_NAO_ATEND_SOLVER"] = (nao_atend_item["NEC_PCS"] - nao_atend_item["QTD_ATENDIDA_SOLVER"]).clip(lower=0)
nao_atend_item["PERC_ATENDIDO_SOLVER"] = np.where(nao_atend_item["NEC_PCS"] > 0, nao_atend_item["QTD_ATENDIDA_SOLVER"] / nao_atend_item["NEC_PCS"], 0)
nao_atend_item["PERC_NAO_ATENDIDO_SOLVER", "QTD_FINAL_VIAVEL_PAI_FILHO", "QTD_AJUSTE_PAI_FILHO", "FATOR_FINAL_VIAVEL_PAI_FILHO", "MOTIVO_AJUSTE_PAI_FILHO"] = 1 - nao_atend_item["PERC_ATENDIDO_SOLVER"]
cols_nao_atend = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "NEC_PCS_ORIGINAL", "NEC_DIRETA_PCS", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "NEC_PCS_CONSOLIDADA", "NEC_PCS", "QTD_ATENDIDA_SOLVER", "QTD_NAO_ATEND_SOLVER", "PERC_ATENDIDO_SOLVER", "PERC_NAO_ATENDIDO_SOLVER"]
nao_atend_item = nao_atend_item[[c for c in cols_nao_atend if c in nao_atend_item.columns]].copy()

# Ajuste pós-solver pai/filho v12: diagnóstico de viabilidade final corrigido e motivo textual ajustado.
ajuste_pai_filho, resumo_pai_filho, capacidade_liberada_pf = calcular_ajuste_pos_solver_pai_filho(
    estrutura_explodida,
    nec_consolidada_solver,
    nao_atend_item,
    plano_producao
)

if not resumo_pai_filho.empty:
    fatores_pf = resumo_pai_filho.rename(columns={"ID_PROD_UNID_FAT_ORIGEM": "ID_PROD_UNID_FAT"})[["MES_REF", "ID_PROD_UNID_FAT", "QTD_PAI_FINAL_VIAVEL_EST", "QTD_REDUCAO_POS_PAI_FILHO", "FATOR_FINAL_VIAVEL_ORIGEM", "FATOR_AJUSTE_SOBRE_SOLVER", "MOTIVO_AJUSTE_PAI_FILHO"]]
    nao_atend_item = nao_atend_item.merge(fatores_pf, on=["MES_REF", "ID_PROD_UNID_FAT"], how="left")
    nao_atend_item["QTD_FINAL_VIAVEL_PAI_FILHO"] = nao_atend_item["QTD_PAI_FINAL_VIAVEL_EST"].fillna(nao_atend_item["QTD_ATENDIDA_SOLVER"])
    nao_atend_item["QTD_FINAL_VIAVEL_PAI_FILHO"] = np.minimum(nao_atend_item["QTD_FINAL_VIAVEL_PAI_FILHO"], nao_atend_item["QTD_ATENDIDA_SOLVER"])
    nao_atend_item["QTD_AJUSTE_PAI_FILHO"] = (nao_atend_item["QTD_ATENDIDA_SOLVER"] - nao_atend_item["QTD_FINAL_VIAVEL_PAI_FILHO"]).clip(lower=0)
    nao_atend_item["FATOR_FINAL_VIAVEL_PAI_FILHO"] = np.where(nao_atend_item["QTD_ATENDIDA_SOLVER"] > TOL, (nao_atend_item["QTD_FINAL_VIAVEL_PAI_FILHO"] / nao_atend_item["QTD_ATENDIDA_SOLVER"]).clip(0, 1), 0)
    nao_atend_item["MOTIVO_AJUSTE_PAI_FILHO"] = nao_atend_item["MOTIVO_AJUSTE_PAI_FILHO"].fillna("NAO_APLICAVEL_SEM_ESTRUTURA_ORIGEM")
    nao_atend_item = nao_atend_item.drop(columns=[c for c in ["QTD_PAI_FINAL_VIAVEL_EST", "QTD_REDUCAO_POS_PAI_FILHO", "FATOR_FINAL_VIAVEL_ORIGEM", "FATOR_AJUSTE_SOBRE_SOLVER"] if c in nao_atend_item.columns])

    fator_plano_pf = fatores_pf[["MES_REF", "ID_PROD_UNID_FAT", "FATOR_FINAL_VIAVEL_ORIGEM", "FATOR_AJUSTE_SOBRE_SOLVER"]]
    plano_producao = plano_producao.merge(fator_plano_pf, on=["MES_REF", "ID_PROD_UNID_FAT"], how="left")
    plano_producao["FATOR_AJUSTE_SOBRE_SOLVER"] = plano_producao["FATOR_AJUSTE_SOBRE_SOLVER"].fillna(1.0)
    plano_producao["FATOR_FINAL_VIAVEL_ORIGEM"] = plano_producao["FATOR_FINAL_VIAVEL_ORIGEM"].fillna(1.0)
    plano_producao["QTD_PRODUZIR_FINAL_VIAVEL_EST"] = plano_producao["QTD_PRODUZIR_SOLVER"] * plano_producao["FATOR_AJUSTE_SOBRE_SOLVER"]
    plano_producao["QTD_REDUCAO_POS_PAI_FILHO"] = (plano_producao["QTD_PRODUZIR_SOLVER"] - plano_producao["QTD_PRODUZIR_FINAL_VIAVEL_EST"]).clip(lower=0)
    plano_producao["HR_LIBERADA_EST"] = np.where(plano_producao["PCS_HORA"] > TOL, plano_producao["QTD_REDUCAO_POS_PAI_FILHO"] / plano_producao["PCS_HORA"], 0)
else:
    nao_atend_item["QTD_FINAL_VIAVEL_PAI_FILHO"] = nao_atend_item["QTD_ATENDIDA_SOLVER"]
    nao_atend_item["QTD_AJUSTE_PAI_FILHO"] = 0
    nao_atend_item["FATOR_FINAL_VIAVEL_PAI_FILHO"] = 1.0
    nao_atend_item["MOTIVO_AJUSTE_PAI_FILHO"] = "SEM_ESTRUTURA_EXPLODIDA"
    plano_producao["FATOR_FINAL_VIAVEL_ORIGEM"] = 1.0
    plano_producao["FATOR_AJUSTE_SOBRE_SOLVER"] = 1.0
    plano_producao["QTD_PRODUZIR_FINAL_VIAVEL_EST"] = plano_producao["QTD_PRODUZIR_SOLVER"]
    plano_producao["QTD_REDUCAO_POS_PAI_FILHO"] = 0
    plano_producao["HR_LIBERADA_EST"] = 0



# Correção v16 — blindagem contra duplicidade na saída do plano.
# A origem do problema era o merge do plano com fatores pai/filho quando o resumo pai/filho tinha mais de uma linha para a mesma chave MES_REF + ID_PROD_UNID_FAT.
# O solver/rastro não estava estourado; a duplicidade ocorria na montagem/exportação do plano e contaminava os resumos.
CHAVE_PLANO_UNICA = [c for c in ['MES_REF','ID_PROD_UNID_FAT','COD_PROD','RODADA','PRIOR_MATPAR','PRIOR_ROT','ALOC_REC','COD_FER_UNID','QTD_PRODUZIR_SOLVER','HR_PRODUZIR_SOLVER'] if c in plano_producao.columns]
QTD_LINHAS_PLANO_ANTES_DEDUP = len(plano_producao)
plano_producao = plano_producao.drop_duplicates(subset=CHAVE_PLANO_UNICA, keep='first').copy()
QTD_LINHAS_PLANO_DEPOIS_DEDUP = len(plano_producao)
print('Correção v16 - linhas plano antes/depois dedupe:', QTD_LINHAS_PLANO_ANTES_DEDUP, QTD_LINHAS_PLANO_DEPOIS_DEDUP)

prod_recurso = plano_producao.groupby("ALOC_REC", as_index=False).agg(HR_PRODUZIR_SOLVER=("HR_PRODUZIR_SOLVER", "sum"), QTD_PRODUZIR_SOLVER=("QTD_PRODUZIR_SOLVER", "sum"), QTD_ITENS=("ID_PROD_UNID_FAT", "nunique")) if not plano_producao.empty else pd.DataFrame(columns=["ALOC_REC", "HR_PRODUZIR_SOLVER", "QTD_PRODUZIR_SOLVER", "QTD_ITENS"])
resumo_recursos = bd_cap_recursos.merge(prod_recurso, on="ALOC_REC", how="left").fillna({"HR_PRODUZIR_SOLVER": 0, "QTD_PRODUZIR_SOLVER": 0, "QTD_ITENS": 0})
resumo_recursos["OCUPACAO_RECURSO_PCT"] = np.where(resumo_recursos["HOR_REC"] > 0, resumo_recursos["HR_PRODUZIR_SOLVER"] / resumo_recursos["HOR_REC"], 0)
resumo_recursos["HR_RECURSO_OCIOSA"] = resumo_recursos["HOR_REC"] - resumo_recursos["HR_PRODUZIR_SOLVER"]
resumo_recursos["ESTOURO_HR_RECURSO"] = np.where(resumo_recursos["HR_PRODUZIR_SOLVER"] > resumo_recursos["HOR_REC"] + TOL, resumo_recursos["HR_PRODUZIR_SOLVER"] - resumo_recursos["HOR_REC"], 0)
resumo_recursos = resumo_recursos[["ALOC_REC", "HOR_REC", "HR_PRODUZIR_SOLVER", "OCUPACAO_RECURSO_PCT", "HR_RECURSO_OCIOSA", "ESTOURO_HR_RECURSO", "QTD_PRODUZIR_SOLVER", "QTD_ITENS"]].sort_values(["OCUPACAO_RECURSO_PCT", "HR_PRODUZIR_SOLVER"], ascending=[False, False])

prod_ferramenta = plano_producao.groupby("COD_FER_UNID", as_index=False).agg(HR_PRODUZIR_SOLVER=("HR_PRODUZIR_SOLVER", "sum"), QTD_PRODUZIR_SOLVER=("QTD_PRODUZIR_SOLVER", "sum"), QTD_ITENS=("ID_PROD_UNID_FAT", "nunique"), QTD_RECURSOS=("ALOC_REC", "nunique")) if not plano_producao.empty else pd.DataFrame(columns=["COD_FER_UNID", "HR_PRODUZIR_SOLVER", "QTD_PRODUZIR_SOLVER", "QTD_ITENS", "QTD_RECURSOS"])
resumo_ferramentas = bd_cap_ferramentas.merge(prod_ferramenta, on="COD_FER_UNID", how="left").fillna({"HR_PRODUZIR_SOLVER": 0, "QTD_PRODUZIR_SOLVER": 0, "QTD_ITENS": 0, "QTD_RECURSOS": 0})
resumo_ferramentas["OCUPACAO_FER_PCT"] = np.where(resumo_ferramentas["HOR_FER"] > 0, resumo_ferramentas["HR_PRODUZIR_SOLVER"] / resumo_ferramentas["HOR_FER"], 0)
resumo_ferramentas["HR_FER_OCIOSA"] = resumo_ferramentas["HOR_FER"] - resumo_ferramentas["HR_PRODUZIR_SOLVER"]
resumo_ferramentas["ESTOURO_HR_FER"] = np.where(resumo_ferramentas["HR_PRODUZIR_SOLVER"] > resumo_ferramentas["HOR_FER"] + TOL, resumo_ferramentas["HR_PRODUZIR_SOLVER"] - resumo_ferramentas["HOR_FER"], 0)
resumo_ferramentas = resumo_ferramentas[["COD_FER_UNID", "HOR_FER", "HR_PRODUZIR_SOLVER", "OCUPACAO_FER_PCT", "HR_FER_OCIOSA", "ESTOURO_HR_FER", "QTD_PRODUZIR_SOLVER", "QTD_ITENS", "QTD_RECURSOS"]].sort_values(["OCUPACAO_FER_PCT", "HR_PRODUZIR_SOLVER"], ascending=[False, False])

cols_auditoria = ["MES_REF", "ALOC_REC", "COD_FER_UNID", "HOR_REC", "HOR_FER", "HOR_CAP"]
auditoria_capacidade = bd_alternativas_item[[c for c in cols_auditoria if c in bd_alternativas_item.columns]].drop_duplicates().sort_values(["ALOC_REC", "COD_FER_UNID"]).copy()

cols_alt = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "ALTERNATIVA_VALIDA", "MOTIVO_INVALIDA"]
alternativas_item = bd_alternativas_item[[c for c in cols_alt if c in bd_alternativas_item.columns]].sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC"]).copy()

cols_rastro = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "RODADA", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "QTD_PENDENTE_ANTES", "QTD_SOLICITADA", "HR_SOLICITADA", "HR_DEMANDA_TOTAL_ALOC_REC", "HR_DEMANDA_TOTAL_COD_FER", "CAP_RESTANTE_ALOC_REC_ANTES", "CAP_RESTANTE_COD_FER_ANTES", "CAP_DISPONIVEL_LINHA", "FATOR_RATEIO_ALOC_REC", "FATOR_RATEIO_COD_FER", "FATOR_RATEIO", "MOTIVO_LIMITANTE", "QTD_PRODUZIR_SOLVER", "HR_PRODUZIR_SOLVER", "QTD_PENDENTE_DEPOIS", "MOTIVO_CORTE", "STATUS_DECISAO"]
rastro_rateio = rastro_rateio[[c for c in cols_rastro if c in rastro_rateio.columns]].copy() if not rastro_rateio.empty else pd.DataFrame(columns=cols_rastro)

qtd_alt_audit = bd_alternativas_item.groupby("ID_PROD_UNID_FAT", as_index=False).agg(QTD_ALTERNATIVAS_AUDITADAS=("ALOC_REC", "count"))
qtd_alt_valid = bd_alternativas_validas.groupby("ID_PROD_UNID_FAT", as_index=False).agg(QTD_ALTERNATIVAS_VALIDAS=("ALOC_REC", "count"))
qtd_alt_usadas = plano_producao.groupby("ID_PROD_UNID_FAT", as_index=False).agg(QTD_ALTERNATIVAS_USADAS=("ALOC_REC", "count"), QTD_ALOC_REC_USADOS=("ALOC_REC", "nunique"), QTD_COD_FER_USADOS=("COD_FER_UNID", "nunique")) if not plano_producao.empty else pd.DataFrame(columns=["ID_PROD_UNID_FAT", "QTD_ALTERNATIVAS_USADAS", "QTD_ALOC_REC_USADOS", "QTD_COD_FER_USADOS"])
qtd_rodadas = rastro_rateio.groupby("ID_PROD_UNID_FAT", as_index=False).agg(QTD_RODADAS_TENTADAS=("RODADA", "nunique"), QTD_ALOC_REC_TENTADOS=("ALOC_REC", "nunique"), QTD_COD_FER_TENTADOS=("COD_FER_UNID", "nunique")) if not rastro_rateio.empty else pd.DataFrame(columns=["ID_PROD_UNID_FAT", "QTD_RODADAS_TENTADAS", "QTD_ALOC_REC_TENTADOS", "QTD_COD_FER_TENTADOS"])

diagnostico_item = nao_atend_item.copy()
for df_merge in [qtd_alt_audit, qtd_alt_valid, qtd_alt_usadas, qtd_rodadas]:
    diagnostico_item = diagnostico_item.merge(df_merge, on="ID_PROD_UNID_FAT", how="left")
for c in ["QTD_ALTERNATIVAS_AUDITADAS", "QTD_ALTERNATIVAS_VALIDAS", "QTD_ALTERNATIVAS_USADAS", "QTD_ALOC_REC_USADOS", "QTD_COD_FER_USADOS", "QTD_RODADAS_TENTADAS", "QTD_ALOC_REC_TENTADOS", "QTD_COD_FER_TENTADOS"]:
    if c in diagnostico_item.columns:
        diagnostico_item[c] = diagnostico_item[c].fillna(0).astype(int)
diagnostico_item["MOTIVO_SALDO_FINAL"] = np.select([diagnostico_item["QTD_NAO_ATEND_SOLVER"] <= TOL, diagnostico_item.get("QTD_ALTERNATIVAS_AUDITADAS", 0).eq(0), diagnostico_item.get("QTD_ALTERNATIVAS_VALIDAS", 0).eq(0), diagnostico_item.get("QTD_ALTERNATIVAS_USADAS", 0).eq(0)], ["ATENDIDO_TOTAL", "SEM_ALTERNATIVA_CADASTRADA", "SEM_ALTERNATIVA_VALIDA", "SEM_ALOCACAO_COM_ALTERNATIVA_VALIDA"], default="SALDO_APOS_RATEIO_E_LIMITES_DE_CAPACIDADE")

uso_alt = plano_producao.groupby(["ID_PROD_UNID_FAT", "ALOC_REC", "COD_FER_UNID", "PRIOR_MATPAR", "PRIOR_ROT"], as_index=False).agg(HR_PRODUZIR_SOLVER=("HR_PRODUZIR_SOLVER", "sum"), QTD_PRODUZIR_SOLVER=("QTD_PRODUZIR_SOLVER", "sum")) if not plano_producao.empty else pd.DataFrame(columns=["ID_PROD_UNID_FAT", "ALOC_REC", "COD_FER_UNID", "PRIOR_MATPAR", "PRIOR_ROT", "HR_PRODUZIR_SOLVER", "QTD_PRODUZIR_SOLVER"])
diagnostico_alternativas = alternativas_item.merge(uso_alt, on=["ID_PROD_UNID_FAT", "ALOC_REC", "COD_FER_UNID", "PRIOR_MATPAR", "PRIOR_ROT"], how="left")
diagnostico_alternativas["HR_PRODUZIR_SOLVER"] = diagnostico_alternativas["HR_PRODUZIR_SOLVER"].fillna(0)
diagnostico_alternativas["QTD_PRODUZIR_SOLVER"] = diagnostico_alternativas["QTD_PRODUZIR_SOLVER"].fillna(0)
diagnostico_alternativas["ALTERNATIVA_USADA"] = diagnostico_alternativas["HR_PRODUZIR_SOLVER"] > TOL

print("Resumo recursos:", len(resumo_recursos))
print("Resumo ferramentas:", len(resumo_ferramentas))
print("Plano produção:", len(plano_producao))
print("Não atendimento:", len(nao_atend_item))
print("Rastro:", len(rastro_rateio))


In [ ]:
# ============================================================
# 06. REALOCAÇÃO DA CAPACIDADE LIBERADA - v16
# ============================================================

# Nesta versão, a capacidade liberada pelo corte pai/filho volta para a máquina (ALOC_REC).
# Para realocar em outro item, o modelo valida a capacidade da máquina e também a capacidade do molde/ferramenta destino (COD_FER_UNID).
# A realocação não fica mais presa ao mesmo molde que gerou a liberação.

# ---------- v16 realocacao ----------
def calcular_realocacao_v16(nao_atend_item, alternativas_item, plano_producao, capacidade_liberada_pf):
    cols_log = [
        'RODADA_REALOC','MES_REF','ID_PROD_UNID_FAT','COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD',
        'PRIOR_MATPAR','PRIOR_ROT','CLASS_ROT','ALOC_REC','COD_FER_UNID','PCS_HORA',
        'QTD_GAP_ANTES_REALOC','HR_MAQ_LIBERADA_DISP_ANTES','HR_FER_DESTINO_DISP_ANTES','HR_DISPONIVEL_REALOC',
        'QTD_POSSIVEL_REALOCAR','QTD_REALOCADA','HR_CONSUMIDA_REALOC','HR_MAQ_LIBERADA_SALDO_DEPOIS',
        'HR_FER_DESTINO_SALDO_DEPOIS','QTD_GAP_DEPOIS_REALOC','STATUS_REALOCACAO','MOTIVO_AJUSTE_PAI_FILHO'
    ]
    if capacidade_liberada_pf.empty:
        return pd.DataFrame(columns=cols_log), pd.DataFrame(), pd.DataFrame()
    na = nao_atend_item.copy(); alts = alternativas_item.copy(); plano = plano_producao.copy(); cap = capacidade_liberada_pf.copy()
    for df in [na, alts, plano, cap]:
        if 'MES_REF' in df.columns:
            df['MES_REF'] = pd.to_datetime(df['MES_REF'], errors='coerce')
    for c in ['NEC_PCS','QTD_FINAL_VIAVEL_PAI_FILHO','QTD_AJUSTE_PAI_FILHO','QTD_ATENDIDA_SOLVER']:
        if c not in na.columns: na[c] = 0.0
        na[c] = to_num(na[c])
    na['QTD_GAP_APOS_PAI_FILHO'] = (na['NEC_PCS'] - na['QTD_FINAL_VIAVEL_PAI_FILHO']).clip(lower=0)
    # Mantem bloqueio: itens cortados por pai/filho nao recebem realocacao nesta rodada, para nao produzir sem componente.
    na['BLOQUEADO_AJUSTE_PAI_FILHO'] = na['QTD_AJUSTE_PAI_FILHO'] > TOL

    if 'ALTERNATIVA_VALIDA' in alts.columns:
        alts['_ALT_VALIDA'] = is_valid_alt(alts['ALTERNATIVA_VALIDA'])
    else:
        alts['_ALT_VALIDA'] = True
    for c in ['PCS_HORA','PRIOR_MATPAR','PRIOR_ROT','HOR_FER','HOR_REC']:
        if c in alts.columns: alts[c] = to_num(alts[c])
    for c in ['HR_PRODUZIR_SOLVER','HOR_FER','HOR_REC']:
        if c in plano.columns: plano[c] = to_num(plano[c])
    for c in ['HR_LIBERADA_EST']:
        cap[c] = to_num(cap[c])

    # Capacidade liberada volta para a maquina, nao fica presa ao molde original.
    pool_maq = cap.groupby(['MES_REF','ALOC_REC'], as_index=False).agg(
        HR_MAQ_LIBERADA_TOTAL=('HR_LIBERADA_EST','sum'),
        QTD_ORIGENS_LIBERARAM=('ID_PROD_UNID_FAT','nunique'),
        QTD_LINHAS_LIBERARAM=('ID_PROD_UNID_FAT','count')
    )

    # Capacidade disponivel por molde destino depois do ajuste pai/filho.
    # HOR_FER total mensal por ferramenta.
    fer_cap_1 = alts.groupby(['MES_REF','COD_FER_UNID'], as_index=False).agg(HOR_FER=('HOR_FER','max')) if 'HOR_FER' in alts.columns else pd.DataFrame(columns=['MES_REF','COD_FER_UNID','HOR_FER'])
    fer_cap_2 = plano.groupby(['MES_REF','COD_FER_UNID'], as_index=False).agg(HOR_FER_PLANO=('HOR_FER','max')) if 'HOR_FER' in plano.columns else pd.DataFrame(columns=['MES_REF','COD_FER_UNID','HOR_FER_PLANO'])
    fer_cap = fer_cap_1.merge(fer_cap_2, on=['MES_REF','COD_FER_UNID'], how='outer')
    fer_cap['HOR_FER'] = fer_cap.get('HOR_FER', 0).fillna(0)
    if 'HOR_FER_PLANO' in fer_cap.columns:
        fer_cap['HOR_FER'] = np.maximum(fer_cap['HOR_FER'], fer_cap['HOR_FER_PLANO'].fillna(0))
    fer_cap = fer_cap[['MES_REF','COD_FER_UNID','HOR_FER']]

    fer_usado_solver = plano.groupby(['MES_REF','COD_FER_UNID'], as_index=False).agg(HR_USADA_SOLVER=('HR_PRODUZIR_SOLVER','sum')) if 'HR_PRODUZIR_SOLVER' in plano.columns else pd.DataFrame(columns=['MES_REF','COD_FER_UNID','HR_USADA_SOLVER'])
    fer_liberada = cap.groupby(['MES_REF','COD_FER_UNID'], as_index=False).agg(HR_LIBERADA_FER=('HR_LIBERADA_EST','sum'))
    fer_pool = fer_cap.merge(fer_usado_solver, on=['MES_REF','COD_FER_UNID'], how='left').merge(fer_liberada, on=['MES_REF','COD_FER_UNID'], how='left').fillna({'HR_USADA_SOLVER':0,'HR_LIBERADA_FER':0})
    fer_pool['HR_FER_USADA_POS_PAI_FILHO'] = (fer_pool['HR_USADA_SOLVER'] - fer_pool['HR_LIBERADA_FER']).clip(lower=0)
    fer_pool['HR_FER_DISPONIVEL_POS_PAI_FILHO'] = (fer_pool['HOR_FER'] - fer_pool['HR_FER_USADA_POS_PAI_FILHO']).clip(lower=0)

    # Candidatos: itens com gap na mesma maquina onde houve liberacao; ferramenta destino tem que ter capacidade.
    item_cols = ['MES_REF','ID_PROD_UNID_FAT','COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD','NEC_PCS','QTD_FINAL_VIAVEL_PAI_FILHO','QTD_GAP_APOS_PAI_FILHO','BLOQUEADO_AJUSTE_PAI_FILHO','MOTIVO_AJUSTE_PAI_FILHO']
    cand = alts.merge(na[item_cols], on=['MES_REF','ID_PROD_UNID_FAT'], how='inner', suffixes=('', '_ITEM'))
    for c in ['COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD']:
        ci = c + '_ITEM'
        if ci in cand.columns:
            cand[c] = cand[c].fillna(cand[ci]) if c in cand.columns else cand[ci]
    cand = cand[(cand['_ALT_VALIDA']) & (cand['PCS_HORA'] > TOL) & (cand['QTD_GAP_APOS_PAI_FILHO'] > TOL) & (~cand['BLOQUEADO_AJUSTE_PAI_FILHO'])].copy()
    cand = cand.merge(pool_maq[['MES_REF','ALOC_REC','HR_MAQ_LIBERADA_TOTAL']], on=['MES_REF','ALOC_REC'], how='inner')
    cand = cand.merge(fer_pool[['MES_REF','COD_FER_UNID','HR_FER_DISPONIVEL_POS_PAI_FILHO']], on=['MES_REF','COD_FER_UNID'], how='left')
    cand['HR_FER_DISPONIVEL_POS_PAI_FILHO'] = cand['HR_FER_DISPONIVEL_POS_PAI_FILHO'].fillna(0)
    cand = cand[cand['HR_FER_DISPONIVEL_POS_PAI_FILHO'] > TOL].copy()

    gap_rem = {tuple(k): float(v) for k, v in na.set_index(['MES_REF','ID_PROD_UNID_FAT'])['QTD_GAP_APOS_PAI_FILHO'].items()}
    bloqueado = {tuple(k): bool(v) for k, v in na.set_index(['MES_REF','ID_PROD_UNID_FAT'])['BLOQUEADO_AJUSTE_PAI_FILHO'].items()}
    hr_maq_rem = {tuple(row[['MES_REF','ALOC_REC']]): float(row['HR_MAQ_LIBERADA_TOTAL']) for _, row in pool_maq.iterrows()}
    hr_fer_rem = {tuple(row[['MES_REF','COD_FER_UNID']]): float(row['HR_FER_DISPONIVEL_POS_PAI_FILHO']) for _, row in fer_pool.iterrows()}

    logs = []
    if not cand.empty:
        cand['_GAP_SORT'] = cand['QTD_GAP_APOS_PAI_FILHO']
        cand = cand.sort_values(['MES_REF','ALOC_REC','PRIOR_MATPAR','PRIOR_ROT','_GAP_SORT','ID_PROD_UNID_FAT'], ascending=[True, True, True, True, False, True])
        rodada = 1
        for _, row in cand.iterrows():
            item_key = (row['MES_REF'], row['ID_PROD_UNID_FAT'])
            maq_key = (row['MES_REF'], row['ALOC_REC'])
            fer_key = (row['MES_REF'], row['COD_FER_UNID'])
            gap_antes = gap_rem.get(item_key, 0.0)
            hr_maq_antes = hr_maq_rem.get(maq_key, 0.0)
            hr_fer_antes = hr_fer_rem.get(fer_key, 0.0)
            pcs_hora = float(row['PCS_HORA'])
            hr_disp = min(hr_maq_antes, hr_fer_antes)
            if gap_antes <= TOL:
                status='NAO_REALOCADO_SEM_GAP_REMANESCENTE'; qtd_possivel=qtd_realoc=hr_consumida=0.0
            elif hr_maq_antes <= TOL:
                status='NAO_REALOCADO_SEM_HORA_MAQUINA_LIBERADA'; qtd_possivel=qtd_realoc=hr_consumida=0.0
            elif hr_fer_antes <= TOL:
                status='NAO_REALOCADO_SEM_HORA_MOLDE_DESTINO'; qtd_possivel=qtd_realoc=hr_consumida=0.0
            elif bloqueado.get(item_key, False):
                status='NAO_REALOCADO_ITEM_BLOQUEADO_PAI_FILHO'; qtd_possivel=qtd_realoc=hr_consumida=0.0
            else:
                qtd_possivel = hr_disp * pcs_hora
                qtd_realoc = min(gap_antes, qtd_possivel)
                hr_consumida = qtd_realoc / pcs_hora if pcs_hora > TOL else 0.0
                if qtd_realoc > TOL:
                    status = 'REALOCADO_TOTAL_GAP' if qtd_realoc >= gap_antes - TOL else 'REALOCADO_PARCIAL_POR_HORA_DISPONIVEL'
                    gap_rem[item_key] = max(0.0, gap_antes - qtd_realoc)
                    hr_maq_rem[maq_key] = max(0.0, hr_maq_antes - hr_consumida)
                    hr_fer_rem[fer_key] = max(0.0, hr_fer_antes - hr_consumida)
                else:
                    status='NAO_REALOCADO_QTD_ZERO'
            logs.append({
                'RODADA_REALOC': rodada,
                'MES_REF': row['MES_REF'],
                'ID_PROD_UNID_FAT': row['ID_PROD_UNID_FAT'],
                'COD_PROD': row.get('COD_PROD'),
                'DESC_PROD': row.get('DESC_PROD'),
                'UNID_PROD': row.get('UNID_PROD'),
                'UNID_FAT': row.get('UNID_FAT'),
                'TIPO_PROD': row.get('TIPO_PROD'),
                'PRIOR_MATPAR': row.get('PRIOR_MATPAR'),
                'PRIOR_ROT': row.get('PRIOR_ROT'),
                'CLASS_ROT': row.get('CLASS_ROT'),
                'ALOC_REC': row['ALOC_REC'],
                'COD_FER_UNID': row['COD_FER_UNID'],
                'PCS_HORA': pcs_hora,
                'QTD_GAP_ANTES_REALOC': gap_antes,
                'HR_MAQ_LIBERADA_DISP_ANTES': hr_maq_antes,
                'HR_FER_DESTINO_DISP_ANTES': hr_fer_antes,
                'HR_DISPONIVEL_REALOC': hr_disp,
                'QTD_POSSIVEL_REALOCAR': qtd_possivel,
                'QTD_REALOCADA': qtd_realoc,
                'HR_CONSUMIDA_REALOC': hr_consumida,
                'HR_MAQ_LIBERADA_SALDO_DEPOIS': hr_maq_rem.get(maq_key, 0.0),
                'HR_FER_DESTINO_SALDO_DEPOIS': hr_fer_rem.get(fer_key, 0.0),
                'QTD_GAP_DEPOIS_REALOC': gap_rem.get(item_key, 0.0),
                'STATUS_REALOCACAO': status,
                'MOTIVO_AJUSTE_PAI_FILHO': row.get('MOTIVO_AJUSTE_PAI_FILHO')
            })
            rodada += 1
    realoc = pd.DataFrame(logs, columns=cols_log)
    realoc_pos = realoc[realoc['QTD_REALOCADA'] > TOL].copy() if not realoc.empty else realoc.copy()

    realoc_item = realoc_pos.groupby(['MES_REF','ID_PROD_UNID_FAT'], as_index=False).agg(
        QTD_REALOCADA_CAP_LIBERADA=('QTD_REALOCADA','sum'),
        HR_CONSUMIDA_REALOC=('HR_CONSUMIDA_REALOC','sum')
    ) if not realoc_pos.empty else pd.DataFrame(columns=['MES_REF','ID_PROD_UNID_FAT','QTD_REALOCADA_CAP_LIBERADA','HR_CONSUMIDA_REALOC'])
    resumo_item = na[['MES_REF','ID_PROD_UNID_FAT','COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD','NEC_PCS','QTD_FINAL_VIAVEL_PAI_FILHO','QTD_GAP_APOS_PAI_FILHO','QTD_AJUSTE_PAI_FILHO','MOTIVO_AJUSTE_PAI_FILHO']].copy()
    resumo_item = resumo_item.merge(realoc_item, on=['MES_REF','ID_PROD_UNID_FAT'], how='left').fillna({'QTD_REALOCADA_CAP_LIBERADA':0,'HR_CONSUMIDA_REALOC':0})
    resumo_item['QTD_FINAL_APOS_REALOCACAO'] = resumo_item['QTD_FINAL_VIAVEL_PAI_FILHO'] + resumo_item['QTD_REALOCADA_CAP_LIBERADA']
    resumo_item['QTD_GAP_FINAL_APOS_REALOCACAO'] = (resumo_item['NEC_PCS'] - resumo_item['QTD_FINAL_APOS_REALOCACAO']).clip(lower=0)
    resumo_item['PERC_GAP_RECUPERADO'] = np.where(resumo_item['QTD_GAP_APOS_PAI_FILHO'] > TOL, resumo_item['QTD_REALOCADA_CAP_LIBERADA'] / resumo_item['QTD_GAP_APOS_PAI_FILHO'], 0)
    resumo_item = resumo_item[resumo_item['QTD_GAP_APOS_PAI_FILHO'] > TOL].sort_values(['QTD_REALOCADA_CAP_LIBERADA','QTD_GAP_APOS_PAI_FILHO'], ascending=[False, False])

    # Resumo interno para validacao (nao vai para Excel final como guia 17)
    resumo_maq = pool_maq.copy()
    uso_maq = realoc_pos.groupby(['MES_REF','ALOC_REC'], as_index=False).agg(
        HR_REAPROVEITADA=('HR_CONSUMIDA_REALOC','sum'),
        QTD_REALOCADA=('QTD_REALOCADA','sum'),
        QTD_ITENS_REALOCADOS=('ID_PROD_UNID_FAT','nunique')
    ) if not realoc_pos.empty else pd.DataFrame(columns=['MES_REF','ALOC_REC','HR_REAPROVEITADA','QTD_REALOCADA','QTD_ITENS_REALOCADOS'])
    resumo_maq = resumo_maq.merge(uso_maq, on=['MES_REF','ALOC_REC'], how='left').fillna({'HR_REAPROVEITADA':0,'QTD_REALOCADA':0,'QTD_ITENS_REALOCADOS':0})
    resumo_maq['HR_MAQ_NAO_REAPROVEITADA'] = (resumo_maq['HR_MAQ_LIBERADA_TOTAL'] - resumo_maq['HR_REAPROVEITADA']).clip(lower=0)
    resumo_maq['PERC_REAPROVEITAMENTO_HR'] = np.where(resumo_maq['HR_MAQ_LIBERADA_TOTAL'] > TOL, resumo_maq['HR_REAPROVEITADA'] / resumo_maq['HR_MAQ_LIBERADA_TOTAL'], 0)
    return realoc, resumo_maq, resumo_item

realocacao_cap_liberada, resumo_realocacao_maquina, resumo_item_realocacao = calcular_realocacao_v16(
    nao_atend_item,
    alternativas_item,
    plano_producao,
    capacidade_liberada_pf
)



In [ ]:

# ============================================================
# 06. VALIDAÇÕES
# ============================================================

nec_total = nao_atend_item["NEC_PCS"].sum()
atend_total = nao_atend_item["QTD_ATENDIDA_SOLVER"].sum()
nao_atend_total = nao_atend_item["QTD_NAO_ATEND_SOLVER"].sum()
dif_fechamento = nec_total - atend_total - nao_atend_total
qtd_rec_estouro = (resumo_recursos["ESTOURO_HR_RECURSO"] > TOL).sum()
qtd_fer_estouro = (resumo_ferramentas["ESTOURO_HR_FER"] > TOL).sum()

validacao = pd.DataFrame({
    "METRICA": ["NEC_PCS_TOTAL", "QTD_ATENDIDA_SOLVER_TOTAL", "QTD_NAO_ATEND_SOLVER_TOTAL", "DIF_FECHAMENTO_DEMANDA", "QTD_ITENS_DEMANDA", "QTD_ITENS_COM_PRODUCAO", "QTD_ITENS_SEM_PRODUCAO", "QTD_ITENS_COM_NAO_ATENDIMENTO", "QTD_RECURSOS", "QTD_RECURSOS_ESTOURO", "ESTOURO_HR_RECURSO_TOTAL", "HOR_REC_TOTAL", "HR_PRODUZIR_SOLVER_RECURSO_TOTAL", "OCUPACAO_GLOBAL_RECURSOS", "QTD_FERRAMENTAS", "QTD_FERRAMENTAS_ESTOURO", "ESTOURO_HR_FER_TOTAL", "HOR_FER_TOTAL", "HR_PRODUZIR_SOLVER_FER_TOTAL", "OCUPACAO_GLOBAL_FERRAMENTAS", "QTD_ALTERNATIVAS_AUDITADAS", "QTD_ALTERNATIVAS_VALIDAS_PRODUCAO", "QTD_ALTERNATIVAS_HOR_CAP_ZERO", "QTD_LINHAS_RASTRO_RATEIO", "QTD_ITENS_DIAGNOSTICO"],
    "VALOR": [nec_total, atend_total, nao_atend_total, dif_fechamento, len(nao_atend_item), (nao_atend_item["QTD_ATENDIDA_SOLVER"] > TOL).sum(), (nao_atend_item["QTD_ATENDIDA_SOLVER"] <= TOL).sum(), (nao_atend_item["QTD_NAO_ATEND_SOLVER"] > TOL).sum(), len(resumo_recursos), qtd_rec_estouro, resumo_recursos["ESTOURO_HR_RECURSO"].sum(), resumo_recursos["HOR_REC"].sum(), resumo_recursos["HR_PRODUZIR_SOLVER"].sum(), resumo_recursos["HR_PRODUZIR_SOLVER"].sum() / resumo_recursos["HOR_REC"].sum() if resumo_recursos["HOR_REC"].sum() > 0 else 0, len(resumo_ferramentas), qtd_fer_estouro, resumo_ferramentas["ESTOURO_HR_FER"].sum(), resumo_ferramentas["HOR_FER"].sum(), resumo_ferramentas["HR_PRODUZIR_SOLVER"].sum(), resumo_ferramentas["HR_PRODUZIR_SOLVER"].sum() / resumo_ferramentas["HOR_FER"].sum() if resumo_ferramentas["HOR_FER"].sum() > 0 else 0, len(bd_alternativas_item), len(bd_alternativas_validas), (bd_alternativas_item["HOR_CAP"] <= TOL).sum(), len(rastro_rateio), len(diagnostico_item)]
})
validacao_checks = pd.DataFrame({"METRICA": ["FECHAMENTO_DEMANDA_OK", "CAPACIDADE_RECURSOS_OK", "CAPACIDADE_FERRAMENTAS_OK"], "VALOR": [abs(dif_fechamento) <= 1e-4, qtd_rec_estouro == 0, qtd_fer_estouro == 0]})
validacao = pd.concat([validacao, pd.DataFrame({"METRICA": ["---"], "VALOR": [""]}), validacao_checks], ignore_index=True)
display(validacao)


# Adiciona bloco de validação da explosão de estrutura dentro da validação geral
validacao = pd.concat([
    validacao,
    pd.DataFrame({"METRICA": ["--- EXPLOSAO_ESTRUTURA ---"], "VALOR": [""]}),
    validacao_explosao
], ignore_index=True)


In [ ]:
# ============================================================
# 07. EXPORTAÇÃO
# ============================================================



# Simplificação v18:
# - cria a base limpa da guia 15 antes de montar o dicionário de exportação;
# - mantém a limpeza da guia 15: não leva os fatores FATOR_FINAL_VIAVEL_ORIGEM e FATOR_AJUSTE_SOBRE_SOLVER;
# - as guias 17_RESUMO_REALOCACAO e 17_RESUMO_REALOCACAO_POOL seguem removidas do arquivo final;
# - adiciona a guia 20_LEGENDA_STATUS com a legenda dos campos de status.
capacidade_liberada_pf_export = capacidade_liberada_pf.drop(
    columns=[col for col in ["FATOR_FINAL_VIAVEL_ORIGEM", "FATOR_AJUSTE_SOBRE_SOLVER"] if col in capacidade_liberada_pf.columns]
).copy()

# Legenda dos campos de status - v18
# Tabela expandida por guia, para facilitar filtro no Excel.
def expandir_status(guias, coluna_status, status_explicacoes):
    linhas = []
    for guia in guias:
        for status, explicacao in status_explicacoes:
            linhas.append({
                "GUIA": guia,
                "COLUNA_STATUS": coluna_status,
                "STATUS": status,
                "EXPLICACAO": explicacao,
            })
    return linhas

linhas_legenda_status = []
linhas_legenda_status += expandir_status(
    ["03_PLANO_PRODUCAO", "07_RASTRO_RATEIO"],
    "STATUS_DECISAO",
    [
        ("ALOCADO_TOTAL_RODADA", "Item atendido totalmente naquela rodada/análise de capacidade."),
        ("ALOCADO_PARCIAL_RATEIO", "Item recebeu produção parcial porque disputou capacidade e sofreu rateio."),
        ("NAO_ALOCADO_CAP_RECURSO_ZERO", "Não produziu porque a máquina/recurso não tinha capacidade disponível."),
        ("NAO_ALOCADO_CAP_FERRAMENTA_ZERO", "Não produziu porque o molde/ferramenta não tinha capacidade disponível."),
    ],
)
linhas_legenda_status += expandir_status(
    ["03_PLANO_PRODUCAO", "07_RASTRO_RATEIO"],
    "MOTIVO_LIMITANTE",
    [
        ("SEM_LIMITACAO", "Havia capacidade suficiente; não houve corte por máquina nem por molde."),
        ("ALOC_REC", "A máquina/recurso foi o limitante; faltou hora da máquina."),
        ("COD_FER_UNID", "O molde/ferramenta foi o limitante; faltou hora do molde."),
        ("ALOC_REC_E_COD_FER", "Máquina e molde limitaram ao mesmo tempo."),
    ],
)
linhas_legenda_status += expandir_status(
    ["03_PLANO_PRODUCAO", "07_RASTRO_RATEIO"],
    "MOTIVO_CORTE",
    [
        ("SEM_CORTE", "A quantidade solicitada foi atendida sem redução."),
        ("CORTE_POR_ALOC_REC", "Quantidade reduzida por falta de hora da máquina/recurso."),
        ("CORTE_POR_COD_FER_UNID", "Quantidade reduzida por falta de hora do molde/ferramenta."),
        ("CORTE_POR_ALOC_REC_E_COD_FER", "Quantidade reduzida porque máquina e molde limitaram juntos."),
    ],
)
linhas_legenda_status += expandir_status(
    ["06_ALTERNATIVAS_ITEM", "09_DIAGNOSTICO_ALTERNATIVAS"],
    "MOTIVO_INVALIDA",
    [
        ("ALTERNATIVA_VALIDA", "Alternativa pode ser usada pelo solver; tem produtividade e capacidade válidas."),
        ("PCS_HORA_ZERO_OU_INVALIDO", "Alternativa inválida porque a produtividade está zerada ou inválida."),
        ("HOR_FER_ZERO_OU_NEGATIVO | HOR_CAP_ZERO_OU_NEGATIVO", "Molde/ferramenta sem capacidade válida; alternativa não deve produzir."),
        ("HOR_REC_ZERO_OU_NEGATIVO | HOR_FER_ZERO_OU_NEGATIVO | HOR_CAP_ZERO_OU_NEGATIVO", "Máquina e/ou molde sem capacidade válida; alternativa inválida."),
    ],
)
linhas_legenda_status += expandir_status(
    ["08_DIAGNOSTICO_ITEM"],
    "MOTIVO_SALDO_FINAL",
    [
        ("ATENDIDO_TOTAL", "Item totalmente atendido pelo solver."),
        ("SALDO_APOS_RATEIO_E_LIMITES_DE_CAPACIDADE", "Item ficou com saldo/gap por limite de capacidade ou rateio."),
        ("SEM_ALOCACAO_COM_ALTERNATIVA_VALIDA", "Item tinha alternativa válida, mas não recebeu produção, normalmente por disputa de capacidade."),
        ("SEM_ALTERNATIVA_VALIDA", "Item não tinha alternativa produtiva válida para o solver."),
    ],
)
linhas_legenda_status += expandir_status(
    ["04_NAO_ATEND_ITEM", "08_DIAGNOSTICO_ITEM", "13_AJUSTE_PAI_FILHO", "14_RESUMO_PAI_FILHO", "16_REALOCACAO_CAP_LIBERADA", "18_RESUMO_ITEM_REALOCACAO"],
    "MOTIVO_AJUSTE_PAI_FILHO",
    [
        ("SEM_AJUSTE_POS_SOLVER", "Produto não sofreu corte adicional pelo ajuste pai/filho."),
        ("PAI_CORTADO_PELO_SOLVER", "O próprio produto pai já foi cortado pelo solver antes da análise dos filhos."),
        ("PAI_NAO_PRODUZIDO_PELO_SOLVER", "O pai não teve produção no solver."),
        ("FILHO_COM_ROTEIRO_LIMITOU_PAI", "Componente/filho com roteiro não foi produzido o suficiente e limitou a quantidade viável do pai."),
        ("FILHO_SEM_ROTEIRO_EXPOSTO_NAO_LIMITA", "Filho/componente sem roteiro é exposto como problema de cadastro, mas não derruba automaticamente o pai."),
        ("SEM_NECESSIDADE_FILHO_POS_CORTE_PAI", "Após o corte do pai, não sobrou necessidade daquele filho naquela relação."),
        ("NAO_APLICAVEL_SEM_ESTRUTURA_ORIGEM", "Item sem estrutura pai/filho aplicável nessa análise."),
    ],
)
linhas_legenda_status += expandir_status(
    ["16_REALOCACAO_CAP_LIBERADA"],
    "STATUS_REALOCACAO",
    [
        ("REALOCADO_TOTAL_GAP", "Realocação cobriu todo o gap do item."),
        ("REALOCADO_PARCIAL_POR_HORA_DISPONIVEL", "Realocação ajudou, mas não zerou o gap porque a hora disponível acabou."),
        ("NAO_REALOCADO_ITEM_BLOQUEADO_PAI_FILHO", "Item não foi realocado porque estava bloqueado pelo ajuste pai/filho; produzir mais poderia gerar item sem componente suficiente."),
        ("NAO_REALOCADO_SEM_GAP_REMANESCENTE", "Item não tinha mais gap para recuperar."),
        ("NAO_REALOCADO_SEM_HORA_MAQUINA_LIBERADA", "Máquina não tinha mais hora liberada disponível para reaproveitar."),
        ("NAO_REALOCADO_SEM_HORA_MOLDE_DESTINO", "Máquina tinha sobra, mas o molde/ferramenta do item destino não tinha capacidade disponível."),
    ],
)
legenda_status = pd.DataFrame(linhas_legenda_status, columns=["GUIA", "COLUNA_STATUS", "STATUS", "EXPLICACAO"])


# Dicionário de guias e colunas - v18
# Gera documentação no próprio Excel final. Cada linha explica uma coluna de uma guia.
GUIAS_EXPORTACAO = {
    "01_RESUMO_RECURSOS": (resumo_recursos, "Resumo por máquina/recurso.", "Validar HOR_REC, ocupação e estouro por máquina."),
    "02_RESUMO_FERRAMENTAS": (resumo_ferramentas, "Resumo por molde/ferramenta.", "Validar HOR_FER, ocupação e estouro por molde."),
    "03_PLANO_PRODUCAO": (plano_producao, "Plano de produção do solver.", "Ver cada alocação positiva por produto, máquina, molde e roteiro."),
    "04_NAO_ATEND_ITEM": (nao_atend_item, "Resumo de atendimento por item.", "Ver necessidade, atendimento e gap por item."),
    "05_AUDITORIA_CAPACIDADE": (auditoria_capacidade, "Auditoria das capacidades.", "Conferir HOR_REC, HOR_FER e HOR_CAP por combinação."),
    "06_ALTERNATIVAS_ITEM": (alternativas_item, "Alternativas produtivas.", "Ver quais roteiros, máquinas e moldes cada item pode usar."),
    "07_RASTRO_RATEIO": (rastro_rateio, "Log do rateio do solver.", "Auditar cada decisão de alocação, corte e capacidade."),
    "08_DIAGNOSTICO_ITEM": (diagnostico_item, "Diagnóstico por item.", "Explicar status de atendimento por item."),
    "09_DIAGNOSTICO_ALTERNATIVAS": (diagnostico_alternativas, "Diagnóstico por alternativa.", "Ver se cada alternativa foi usada e quanto produziu."),
    "10_ESTRUTURA_EXPLODIDA": (estrutura_explodida, "Explosão de estrutura.", "Ver como pais geraram necessidade de componentes."),
    "11_NEC_EXPLODIDA_COMPONENTES": (nec_explodida_componentes, "Necessidade explodida por componente.", "Ver necessidade gerada pela estrutura por componente."),
    "12_NEC_CONSOLIDADA_SOLVER": (nec_consolidada_solver, "Necessidade final enviada ao solver.", "Ver demanda direta + explosão líquida."),
    "13_AJUSTE_PAI_FILHO": (ajuste_pai_filho, "Log pai/filho.", "Auditar se os componentes sustentam o pai."),
    "14_RESUMO_PAI_FILHO": (resumo_pai_filho, "Resumo pai/filho por origem.", "Ver quantidade viável e corte por produto pai."),
    "15_CAPACIDADE_LIBERADA_PF": (capacidade_liberada_pf_export, "Horas liberadas pelo pai/filho.", "Converter corte em peças para horas liberadas."),
    "16_REALOCACAO_CAP_LIBERADA": (realocacao_cap_liberada, "Log da realocação.", "Ver como horas liberadas da máquina foram usadas em itens com gap, validando o molde destino."),
    "18_RESUMO_ITEM_REALOCACAO": (resumo_item_realocacao, "Resumo final por item após realocação.", "Ver gap antes, quantidade recuperada e gap final."),
    "20_LEGENDA_STATUS": (legenda_status, "Legenda dos campos de status.", "Consultar o significado de cada status usado nas guias do solver."),
}

EXPLICACOES_COLUNAS = {

    "GUIA": ("Guia do Excel.", "Nome da guia onde o status aparece ou à qual a legenda se aplica."),
    "COLUNA_STATUS": ("Coluna de status.", "Nome do campo de status, motivo ou classificação documentado."),
    "STATUS": ("Valor do status.", "Valor textual que pode aparecer na coluna de status indicada."),
    "EXPLICACAO": ("Explicação do status.", "Significado prático do status para leitura, auditoria e apresentação ao cliente."),
    "MES_REF": ("Mês de referência.", "Mês em que necessidade, capacidade ou decisão foi processada."),
    "ALOC_REC": ("Máquina/recurso.", "Recurso produtivo onde a produção consome horas. Na v16, a sobra volta para esta máquina."),
    "COD_FER_UNID": ("Molde/ferramenta.", "Ferramenta exigida pelo produto. A realocação só ocorre se o molde destino tiver capacidade."),
    "HOR_REC": ("Horas disponíveis da máquina.", "Limite mensal de horas do recurso."),
    "HOR_FER": ("Horas disponíveis do molde.", "Limite mensal de horas da ferramenta/molde."),
    "HOR_CAP": ("Capacidade da alternativa.", "Menor valor entre HOR_REC e HOR_FER, usado para auditoria da alternativa."),
    "HR_PRODUZIR_SOLVER": ("Horas alocadas pelo solver.", "Horas efetivamente consumidas pela produção planejada."),
    "QTD_PRODUZIR_SOLVER": ("Quantidade planejada pelo solver.", "Quantidade que o solver conseguiu encaixar respeitando capacidade e prioridade."),
    "ESTOURO_HR_RECURSO": ("Estouro da máquina.", "Deve ser zero; mostra quanto ultrapassou HOR_REC se houver erro."),
    "ESTOURO_HR_FER": ("Estouro do molde.", "Deve ser zero; mostra quanto ultrapassou HOR_FER se houver erro."),
    "ID_PROD_UNID_FAT": ("Chave do item no solver.", "Liga o item entre plano, diagnóstico, alternativas e realocação."),
    "COD_PROD": ("Código do produto.", "Código do item/produto na base."),
    "DESC_PROD": ("Descrição do produto.", "Nome do item para leitura e apresentação."),
    "PCS_HORA": ("Peças por hora.", "Produtividade da alternativa; converte hora em quantidade e quantidade em hora."),
    "QTD_GAP_ANTES_REALOC": ("Gap antes da realocação.", "Quantidade que faltava produzir antes da segunda rodada."),
    "HR_MAQ_LIBERADA_DISP_ANTES": ("Sobra da máquina antes.", "Horas liberadas disponíveis na máquina antes da tentativa."),
    "HR_FER_DESTINO_DISP_ANTES": ("Sobra do molde destino antes.", "Horas disponíveis no molde do produto que receberá realocação."),
    "HR_DISPONIVEL_REALOC": ("Hora disponível para realocar.", "Menor valor entre sobra da máquina e sobra do molde destino."),
    "QTD_REALOCADA": ("Quantidade realocada.", "Quantidade efetivamente recuperada com horas liberadas."),
    "HR_CONSUMIDA_REALOC": ("Horas consumidas na realocação.", "QTD_REALOCADA dividida por PCS_HORA."),
    "QTD_GAP_DEPOIS_REALOC": ("Gap depois da realocação.", "Quantidade ainda faltante após a tentativa."),
    "STATUS_REALOCACAO": ("Status da realocação.", "Indica se recuperou total, parcial ou não realocou."),
    "NEC_PCS": ("Necessidade total no solver.", "Demanda final que o solver tenta atender."),
    "QTD_ATENDIDA_SOLVER": ("Quantidade atendida.", "Quantidade produzida pelo solver para o item."),
    "QTD_NAO_ATEND_SOLVER": ("Quantidade não atendida.", "NEC_PCS menos QTD_ATENDIDA_SOLVER."),
    "MOTIVO_AJUSTE_PAI_FILHO": ("Motivo pai/filho.", "Explica se houve corte por componente, corte pelo solver ou outro motivo."),
    "QTD_REDUCAO_POS_PAI_FILHO": ("Corte pós pai/filho.", "Diferença entre quantidade planejada e quantidade final viável."),
    "HR_LIBERADA_EST": ("Horas liberadas estimadas.", "QTD_REDUCAO_POS_PAI_FILHO dividida por PCS_HORA."),
}

def explicar_coluna(coluna):
    if coluna in EXPLICACOES_COLUNAS:
        return EXPLICACOES_COLUNAS[coluna]
    if str(coluna).startswith('QTD_'):
        return ("Quantidade do cálculo.", "Campo quantitativo usado para demanda, produção, corte, gap ou contagem.")
    if str(coluna).startswith('HR_'):
        return ("Horas do cálculo.", "Campo em horas usado para capacidade, consumo, sobra ou realocação.")
    if str(coluna).startswith('FATOR_') or 'PERC' in str(coluna) or str(coluna).endswith('_PCT'):
        return ("Percentual/fator.", "Campo proporcional usado para atendimento, ocupação, rateio ou viabilidade.")
    return ("Campo de apoio.", "Campo usado para identificação, contexto, diagnóstico ou rastreabilidade.")

linhas_dic = []
for nome_guia, (df_guia, resumo, uso) in GUIAS_EXPORTACAO.items():
    for coluna in df_guia.columns:
        simples, completa = explicar_coluna(coluna)
        linhas_dic.append({
            "GUIA": nome_guia,
            "RESUMO_DA_GUIA": resumo,
            "PARA_QUE_SERVE": uso,
            "COLUNA": coluna,
            "EXPLICACAO_SIMPLES": simples,
            "EXPLICACAO_COMPLETA": completa
        })
dicionario_guias = pd.DataFrame(linhas_dic)

OUTPUT_SOLVER_FILE.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUTPUT_SOLVER_FILE, engine="openpyxl") as writer:
    resumo_recursos.to_excel(writer, sheet_name="01_RESUMO_RECURSOS", index=False)
    resumo_ferramentas.to_excel(writer, sheet_name="02_RESUMO_FERRAMENTAS", index=False)
    plano_producao.to_excel(writer, sheet_name="03_PLANO_PRODUCAO", index=False)
    nao_atend_item.to_excel(writer, sheet_name="04_NAO_ATEND_ITEM", index=False)
    auditoria_capacidade.to_excel(writer, sheet_name="05_AUDITORIA_CAPACIDADE", index=False)
    alternativas_item.to_excel(writer, sheet_name="06_ALTERNATIVAS_ITEM", index=False)
    rastro_rateio.to_excel(writer, sheet_name="07_RASTRO_RATEIO", index=False)
    diagnostico_item.to_excel(writer, sheet_name="08_DIAGNOSTICO_ITEM", index=False)
    diagnostico_alternativas.to_excel(writer, sheet_name="09_DIAGNOSTICO_ALTERNATIVAS", index=False)
    estrutura_explodida.to_excel(writer, sheet_name="10_ESTRUTURA_EXPLODIDA", index=False)
    nec_explodida_componentes.to_excel(writer, sheet_name="11_NEC_EXPLODIDA_COMPONENTES", index=False)
    nec_consolidada_solver.to_excel(writer, sheet_name="12_NEC_CONSOLIDADA_SOLVER", index=False)
    ajuste_pai_filho.to_excel(writer, sheet_name="13_AJUSTE_PAI_FILHO", index=False)
    resumo_pai_filho.to_excel(writer, sheet_name="14_RESUMO_PAI_FILHO", index=False)
    capacidade_liberada_pf_export.to_excel(writer, sheet_name="15_CAPACIDADE_LIBERADA_PF", index=False)
    realocacao_cap_liberada.to_excel(writer, sheet_name="16_REALOCACAO_CAP_LIBERADA", index=False)
    resumo_item_realocacao.to_excel(writer, sheet_name="18_RESUMO_ITEM_REALOCACAO", index=False)
    # v18: exportar também a guia 19_DICIONARIO_GUIAS com documentação das guias e colunas.
    dicionario_guias.to_excel(writer, sheet_name="19_DICIONARIO_GUIAS", index=False)
    # v18: guia dedicada com a legenda dos status e motivos usados no solver.
    legenda_status.to_excel(writer, sheet_name="20_LEGENDA_STATUS", index=False)

# Formatação básica
wb = load_workbook(OUTPUT_SOLVER_FILE)
header_fill = PatternFill("solid", fgColor="1F4E78")
header_font = Font(color="FFFFFF", bold=True)
for ws in wb.worksheets:
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for col_cells in ws.columns:
        max_len = 0
        col_letter = col_cells[0].column_letter
        for cell in col_cells[:1000]:
            if cell.value is not None:
                max_len = max(max_len, min(len(str(cell.value)), 60))
        ws.column_dimensions[col_letter].width = min(max(max_len + 2, 10), 45)
wb.save(OUTPUT_SOLVER_FILE)

print("Arquivo gerado:", OUTPUT_SOLVER_FILE)
print("Guias exportadas:", wb.sheetnames)


In [ ]:
timer.finalizar()
